# X10D2 — Project One: Group Challenge

## ExoticMeatMarkets.com — Full Dallas Expansion

Over the past several weeks you've built a complete delivery optimization pipeline:
- Geocoding addresses and building road network graphs (X9D1)
- Greedy allocation by margin and TSP routing (X9D2)
- Refactoring into reusable functions and comparing truck scenarios (X10D1)

**Today the training wheels come off.**

ExoticMeatMarkets.com has received orders from **29 Dallas restaurants** — up from 10.
The supplier has also expanded: **5 trucks** are now available to rent,
each with a different cost structure and capacity.
You may also deploy **up to two trucks simultaneously** on a given day.

### Company Headquarters — Dallas Love Field
All routes begin and end at the ExoticMeatMarkets.com distribution hub:

> **8333 George Coker Cir, Dallas, TX** *(Dallas Love Field)*

The Cuy arrives here each morning. Your truck(s) depart from this address,
complete the delivery route, and return. **Route distance must include the
leg from HQ to the first stop and the return leg from the last stop back to HQ.**

### Real-World Variability
On actual delivery days, supply levels and restaurant demand shift constantly.
**Your solution must be dynamic** — it should work correctly for any combination
of supply availability and bid prices, not just the values in `restaurants_full.xlsx`.
Your group will be given a specific supply scenario at the start of X11D2;
your notebook should be ready to re-run with new inputs in minutes.

Your group's goal: **build a solution that dynamically identifies the most profitable
1- or 2-truck configuration given any supply level, demand, and bid prices.**

---

## Project Deliverables

| Item | Due |
|------|-----|
| Group work session (in class) | Today, X10D2 — last 30 minutes |
| Full work day | X11D1 (Tuesday) |
| **Final notebook submitted to GitHub + Canvas** | **X11D2 (Thursday), start of class** |

**One submission per group.** Submit a link to your GitHub repository on Canvas.

---

## The Data Files

You have two new files for this project:

### `restaurants_full.xlsx`
29 restaurants across Dallas with four columns:
- `RESTAURANT` — name
- `ADDRESS` — street address (Dallas, TX)
- `ENTREES_REQUESTED` — how many entrees each restaurant ordered
- `ENTREE_BID_PRICE` — what that restaurant is willing to pay per entree

**Bid prices range from $35 to $95.** Your supply cost remains **$28.00 per entree**.
Margins are tighter than your X9 set — restaurant selection and routing decisions matter more.

⚠️ **These values represent one possible day.** On delivery day (X11D2) your instructor
will announce the actual supply available and may adjust bid prices. Your notebook
must accept `DAILY_SUPPLY_CAP` as a variable — never hardcode it.

### `trucks.csv`
Five trucks available to rent with four columns:
- `truck_name`
- `fixed_cost` — flat daily fee regardless of distance
- `per_mile_cost` — variable cost per mile driven
- `capacity` — maximum entrees the truck can carry

**No single truck dominates.** A truck with low fixed cost may have high per-mile cost.
A high-capacity truck costs more to deploy. You must run scenarios to find the winner.


## Constraints and Rules

### Starting Point — Company HQ
Every route starts **and ends** at:
> **8333 George Coker Cir, Dallas, TX** (Dallas Love Field)

Geocode this address exactly once. Snap it to its nearest OSMnx node.
The first leg of every route is HQ → first restaurant stop.
The last leg is final restaurant stop → HQ.
Total miles must include both of these legs.

### Supply Cap — Variable by Day
ExoticMeatMarkets.com can supply up to **165 entrees** in the default scenario,
across both trucks combined if you deploy two. **However, the actual supply
available on delivery day will be announced by your instructor at the start of X11D2
and may differ.** Your notebook must treat `DAILY_SUPPLY_CAP` as a variable
that can be changed in a single cell without breaking anything downstream.

### Single-Truck Rules
- You pick **one truck** from `trucks.csv`
- That truck's `capacity` caps how many entrees it can physically carry
- Effective cap = `min(DAILY_SUPPLY_CAP, truck.capacity)`
- Route the truck through your selected restaurants (greedy TSP, as before)
- Compute the full P&L

### Two-Truck Rules (optional — but potentially more profitable)
- You pick **two different trucks** from `trucks.csv`
- You **split the 29 restaurants** between the two trucks however you like
- Each truck gets its own allocation and its own route
- Each truck pays its own fixed + variable costs
- **Combined entrees across both trucks ≤ 165** (the daily supply cap still applies)
- **Each truck must serve at least 3 restaurants** (a one-stop delivery doesn't justify the truck)
- Net profit = Truck 1 net profit + Truck 2 net profit

### General Rules
- A restaurant cannot be served by both trucks
- You cannot serve more entrees than a restaurant requested
- Partial fills are allowed (serve fewer entrees than requested if supply runs out)
- All routes use real Dallas road distances (OSMnx graph from X9D1/X10D1)


## Scoring Rubric

| Category | Points | Description |
|----------|--------|-------------|
| **Correctness** | 25 | P&L math is right; HQ-anchored routes; constraints respected; assertions pass |
| **Code Quality** | 20 | Functions used appropriately; DRY (don't repeat your code); readable; docstrings |
| **Recommendation** | 15 | Produces clear recommendations with numbers to back it up |
| **Validation** | 10 | At least 5 assert-based checks covering key constraints and math |
| **Creativity** | 30 | A creative approach that distinguishes your solution from others (didn't just run it through Claude) |
| **Total** | 100 | |

### Correctness details
Your notebook must produce a **final P&L table** showing:
- Revenue, supply cost, gross profit
- Truck fixed cost, truck variable cost (miles × per_mile_cost)
- Net profit

If deploying two trucks, show the P&L for each truck and a combined summary row.
Miles must include the HQ departure and return legs.

### Scenario Exploration details
Your notebook must show **at least three distinct scenarios** you evaluated:
- At minimum: each single-truck option you seriously considered
- At least one two-truck scenario
- At least two different supply cap values tested (e.g., 100, 165)
- A comparison table at the end showing net profit for each scenario

### Recommendation
A markdown cell at the end of your notebook titled **"Our Recommendation"** that:
1. States which truck(s) you recommend and why
2. Cites specific numbers (net profit, miles driven, key restaurants served/skipped)
3. Acknowledges any trade-offs or assumptions in your approach
4. Explains how your recommendation would change at different supply levels

### Creativity details
There is no single formula here — this category rewards groups that go beyond the
scaffold and bring something original. Examples of what earns creativity points:
- A novel restaurant-split strategy for two trucks with a clear rationale
- A function that automatically selects the best 1-vs-2-truck configuration
  given any supply level (no manual testing required)
- A visualization that makes the trade-offs immediately obvious
- A sensitivity analysis showing at which supply level the optimal truck changes
- Any other approach that makes your solution more useful on a real delivery day

The best solutions will be ones your instructor could actually hand to the delivery
driver on the morning of X11D2 and get a confident answer in seconds.


## Hints and Suggested Approach

### Start from X10D1
Your `allocate_supply()`, `build_route()`, and `calculate_pnl()` functions from X10D1
are already the right building blocks. Copy them into your project notebook.

The main changes you'll need:
1. Load `restaurants_full.xlsx` instead of `restaurants_x9.xlsx`
2. Load truck configs from `trucks.csv` instead of hardcoded dicts
3. Add HQ as a fixed start/end node to `build_route()`
4. Make `DAILY_SUPPLY_CAP` easy to change — set it once at the top of your notebook
5. Loop over truck options to compare scenarios; consider automating the selection

### Geocoding HQ
Add the HQ address to your geocoding step as a special row, or geocode it separately.
Snap it to its OSMnx node just like a restaurant. Then modify `build_route()` so
it always starts from the HQ node and returns to it at the end.

```python
HQ_ADDRESS = '8333 George Coker Cir, Dallas, TX'
# Geocode once and store the node:
# hq_lat, hq_lon = geocode_address(HQ_ADDRESS)
# HQ_NODE = ox.nearest_nodes(G, hq_lon, hq_lat)
```

### Dynamic supply cap
Set this once at the top of your notebook and never reference a literal number again:
```python
DAILY_SUPPLY_CAP = 165   # ← change this one line on delivery day
```
Every downstream function call should pass `DAILY_SUPPLY_CAP` as a variable.
On X11D2 your instructor may announce a different number — you should be able
to update it and re-run the entire notebook in under a minute.

### Loading trucks from CSV
```python
import pandas as pd

trucks_df = pd.read_csv('trucks.csv')
print(trucks_df)

# Convert each row to a dict — same structure as X10D1's TRUCK_A / TRUCK_B
truck_list = trucks_df.to_dict(orient='records')
# truck_list[0] looks like:
# {'truck_name': 'Sparrow', 'fixed_cost': 75.0, 'per_mile_cost': 2.5, 'capacity': 60}
```

### Auto-selecting the best configuration (Creativity opportunity)
Rather than manually inspecting a comparison table, consider writing a function
that accepts `DAILY_SUPPLY_CAP` and returns the best truck (or truck pair)
automatically:
```python
def find_best_configuration(df, truck_list, supply_cap, G, HQ_NODE):
    """
    Evaluate all single-truck scenarios and a selection of two-truck scenarios.
    Return the configuration with the highest net profit.
    """
    # TODO: your group designs this
    pass
```

### Two-truck split strategies to consider
Think about what makes a good split — these are just starting points:
- Geographic split (north Dallas vs south Dallas by latitude)
- Price tier split (high bid-price restaurants on one truck, casual on the other)
- Margin split (top N restaurants by margin on Truck 1, rest on Truck 2)
- Distance-from-HQ split (close restaurants on a small cheap truck, far ones on a larger one)

There is no one right answer. Part of the challenge is figuring out what to test.

### Watch out for
- **Geocoding:** `restaurants_full.xlsx` has 29 addresses plus HQ. Geocoding all 30 from scratch
  will take ~60 seconds. Use the same cache-to-disk pattern from X9D1/X10D1.
  Name your cache file `restaurants_full_geocoded.xlsx`.
- **Graph:** You already have `dallas_drive.graphml`. Reuse it — don't re-download.
- **Column name:** The bid price column is `ENTREE_BID_PRICE` (same as X9). ✓


## Section 1 — Starter Scaffold

The cell below gives you a working starting point.
Complete the `# TODO` sections as a group.

You are **not** required to follow this structure — if your group has a better
approach, use it. But make sure your final notebook covers all rubric categories.


In [575]:
import os, time
import pandas as pd
import osmnx as ox
import networkx as nx
import folium
from folium.plugins import AntPath
from geopy.geocoders import Nominatim

# ── Constants ─────────────────────────────────────────────────────────────────
SUPPLY_COST_PER_ENTREE = 52.00   # what ExoticMeatMarkets.com charges you
DAILY_SUPPLY_CAP       = 427     # ← UPDATE THIS on delivery day (X11D2)
METERS_PER_MILE        = 1609.34

# Company headquarters — all routes start and end here
HQ_ADDRESS = '8333 George Coker Cir, Dallas, TX'   # Dallas Love Field

print(f'Supply cost per entree: ${SUPPLY_COST_PER_ENTREE:.2f}')
print(f'Daily supply cap:       {DAILY_SUPPLY_CAP} entrees')
print(f'HQ address:             {HQ_ADDRESS}')


Supply cost per entree: $52.00
Daily supply cap:       427 entrees
HQ address:             8333 George Coker Cir, Dallas, TX


## Phase 2: Data Geocoding & Graph Initialization

Use Nominatim to geocode the company HQ and all restaurant addresses, then load the OpenStreetMap road network graph that was built in X10D1. All addresses are cached to disk to avoid redundant API calls.

### Geocoding HQ
The cell below geocodes HQ and snaps it to the road network.
Run it once — the result is stored in `HQ_NODE` and reused by `build_route()`.
If you already have the cached graph from X10D1, HQ geocoding takes about 2 seconds.


In [576]:
# ── Geocode HQ (run once; result stored in HQ_NODE after graph loads) ──────────
HQ_GEO_FILE = 'hq_geocoded.csv'

if os.path.exists(HQ_GEO_FILE):
    hq_row  = pd.read_csv(HQ_GEO_FILE).iloc[0]
    HQ_LAT  = float(hq_row['LATITUDE'])
    HQ_LON  = float(hq_row['LONGITUDE'])
    print(f'HQ coords loaded from cache: ({HQ_LAT:.5f}, {HQ_LON:.5f})')
else:
    print('Geocoding HQ address...')
    _geo = Nominatim(user_agent='cuy-delivery-hq')
    time.sleep(2)
    _loc = _geo.geocode(HQ_ADDRESS, timeout=10)
    assert _loc is not None, f'Could not geocode HQ: {HQ_ADDRESS}'
    HQ_LAT, HQ_LON = _loc.latitude, _loc.longitude
    pd.DataFrame([{'ADDRESS': HQ_ADDRESS,
                   'LATITUDE': HQ_LAT,
                   'LONGITUDE': HQ_LON}]).to_csv(HQ_GEO_FILE, index=False)
    print(f'HQ geocoded and cached: ({HQ_LAT:.5f}, {HQ_LON:.5f})')

# HQ_NODE is set after the graph loads (see code_graph cell)


HQ coords loaded from cache: (32.85373, -96.84692)


### Load Restaurant and Truck Data

In [577]:
# ── Load restaurant data ───────────────────────────────────────────────────────
df_raw = pd.read_excel('restaurants_presentation.xlsx')
print(f'Restaurants loaded: {len(df_raw)}')
print(f'Total entrees demanded: {df_raw["ENTREES_REQUESTED"].sum()}')
print(f'Bid price range: ${df_raw["ENTREE_BID_PRICE"].min()} – ${df_raw["ENTREE_BID_PRICE"].max()}')
display(df_raw.head())

# ── Load truck configs ─────────────────────────────────────────────────────────
trucks_df  = pd.read_csv('trucks_presentation.csv')
truck_list = trucks_df.to_dict(orient='records')
print(f'\nTrucks available: {len(truck_list)}')
display(trucks_df)


Restaurants loaded: 32
Total entrees demanded: 589
Bid price range: $42 – $95


,RESTAURANT,ADDRESS,ENTREES_REQUESTED,ENTREE_BID_PRICE
0,The French Room and Cuy Porch,"1321 Commerce St, Dallas, TX",12,88
1,Pappas Bros. Steak and Cuy House,"10477 Lombardy Ln, Dallas, TX",12,90
2,Truluck's Ocean's Finest Seafood Crab and Cuy,"2401 McKinney Avenue, Dallas, TX",20,78
3,Al Biernat's Fine Cuy,"4217 Oak Lawn Avenue, Dallas, TX",14,95
4,Sweet Georgia Cuy,"2840 E Ledbetter Dr, Dallas, TX",20,48



Trucks available: 5


,truck_name,fixed_cost,per_mile_cost,capacity
0,Kite,55,3.50,75
1,Heron,120,2.25,120
2,Pelican,160,1.50,150
3,Albatross,240,1.25,220
4,Condor,185,2.75,999999


### Geocode All Addresses with Caching
Address geocoding uses Nominatim and caches results to Excel to avoid re-geocoding on reruns.

In [578]:
# ── Geocode addresses (cached) ─────────────────────────────────────────────────
GEO_FILE = 'restaurants_presentation_geocoded.xlsx'

if os.path.exists(GEO_FILE):
    geo_df = pd.read_excel(GEO_FILE)
    print(f'Loaded geocoded coords from {GEO_FILE}')
else:
    print('Geocoding 32 addresses — this takes ~60 seconds (one time only)...')
    geolocator = Nominatim(user_agent='cuy-delivery-project1')

    def geocode_address(address):
        try:
            time.sleep(2)
            loc = geolocator.geocode(address, timeout=10)
            return (loc.latitude, loc.longitude) if loc else (None, None)
        except Exception as e:
            print(f'  Error: {address} — {e}')
            return (None, None)

    geo_df = df_raw[['RESTAURANT', 'ADDRESS']].copy()
    geo_df['LATITUDE'], geo_df['LONGITUDE'] = zip(
        *geo_df['ADDRESS'].apply(geocode_address)
    )
    geo_df.to_excel(GEO_FILE, index=False)
    print(f'Geocoding complete — saved to {GEO_FILE}')

df_raw = df_raw.merge(geo_df[['RESTAURANT', 'LATITUDE', 'LONGITUDE']],
                      on='RESTAURANT', how='left')

# Sanity check
bad = df_raw[df_raw['LATITUDE'].isna() | df_raw['LONGITUDE'].isna()]
if not bad.empty:
    print(f'⚠️  {len(bad)} restaurants failed geocoding:')
    display(bad[['RESTAURANT', 'ADDRESS']])
else:
    print(f'✅ All {len(df_raw)} restaurants geocoded successfully')


Loaded geocoded coords from restaurants_presentation_geocoded.xlsx
✅ All 32 restaurants geocoded successfully


### Load OSMnx Road Network Graph
The graph contains all streets and intersections in Dallas. Restaurants are snapped to the nearest graph node for routing calculations.

In [579]:
# ── Load road network graph (reuse existing file if present) ───────────────────
GRAPH_FILE = 'dallas_drive.graphml'

if os.path.exists(GRAPH_FILE):
    G = ox.load_graphml(GRAPH_FILE)
    print(f'Graph loaded: {len(G.nodes):,} nodes, {len(G.edges):,} edges')
else:
    print('Graph not found — downloading from OpenStreetMap (3–5 min, one time only)...')
    origin = (df_raw['LATITUDE'].mean(), df_raw['LONGITUDE'].mean())
    ox.settings.use_cache = True
    G = ox.graph_from_point(origin, dist=32186, network_type='drive', simplify=True)
    ox.save_graphml(G, filepath=GRAPH_FILE)
    print(f'Graph saved to {GRAPH_FILE}')

# Snap restaurants to nearest graph nodes
df_raw['NODE_ID'] = df_raw.apply(
    lambda r: ox.nearest_nodes(G, r['LONGITUDE'], r['LATITUDE']), axis=1
)

# Snap HQ to its nearest node — used as route start/end
HQ_NODE = ox.nearest_nodes(G, HQ_LON, HQ_LAT)
print(f'HQ snapped to node: {HQ_NODE}')

# Add margin columns
df_raw['NET_MARGIN']      = df_raw['ENTREE_BID_PRICE'] - SUPPLY_COST_PER_ENTREE
df_raw['GROSS_POTENTIAL'] = df_raw['NET_MARGIN'] * df_raw['ENTREES_REQUESTED']

print('\nRestaurants by net margin (descending):')
display(df_raw[['RESTAURANT', 'ENTREES_REQUESTED', 'ENTREE_BID_PRICE',
                'NET_MARGIN', 'GROSS_POTENTIAL']].sort_values('NET_MARGIN', ascending=False))


Graph loaded: 136,809 nodes, 340,994 edges
HQ snapped to node: 4495043100

Restaurants by net margin (descending):


,RESTAURANT,ENTREES_REQUESTED,ENTREE_BID_PRICE,NET_MARGIN,GROSS_POTENTIAL
3,Al Biernat's Fine Cuy,14,95,43.0,602.0
5,The Capital Cuy,15,92,40.0,600.0
1,Pappas Bros. Steak and Cuy House,12,90,38.0,456.0
0,The French Room and Cuy Porch,12,88,36.0,432.0
19,Al Biernat's Fine Cuy North,14,88,36.0,504.0
11,Pyramid Contemporary Cuy,10,85,33.0,330.0
15,La Stella Cuyhouse,18,84,32.0,576.0
6,Café South Pacific Cuy,16,82,30.0,480.0
14,Tei Tei Cuy Robata,15,80,28.0,420.0
10,Gorji Cuy,16,78,26.0,416.0


## Phase 3: Core Optimization Functions

Three reusable functions that power all scenarios:
- `allocate_supply()`: Select restaurants to serve based on profit margin, respecting supply and truck capacity constraints
- `build_route()`: Order stops using greedy nearest-neighbor TSP from HQ and back
- `calculate_pnl()`: Compute revenue, costs, and net profit for a truck deployment

In [580]:
# ── Core functions from X10D1 — updated for HQ-anchored routes ───────────────
# Copy your three functions here and make the modifications noted below.

def allocate_supply(df, supply_cap, truck_capacity, hq_node=None, distance_cache=None, per_mile_cost=0):
    """Allocate entrees to restaurants by descending NET PROFIT AFTER DISTANCE COSTS.
    Respects both the daily supply cap and the truck's physical capacity.
    
    Parameters
    ----------
    df : DataFrame with restaurant data
    supply_cap : daily supply capacity
    truck_capacity : truck carrying capacity
    hq_node : node ID of HQ (for distance calculation)
    distance_cache : dict of (from_node, to_node) → distance in meters
    per_mile_cost : truck cost per mile (for calculating distance cost)
    
    Returns
    -------
    DataFrame of allocated stops, sorted by adjusted profitability (profit - distance cost)
    """
    # The truck can only carry as many as both constraints allow
    effective_cap = min(supply_cap, truck_capacity)

    # Calculate profit and distance-adjusted profitability
    df = df.copy()
    df['NET_PROFIT'] = df['ENTREES_REQUESTED'] * df['NET_MARGIN']
    
    # Calculate distance cost if cache and HQ node provided
    if hq_node is not None and distance_cache is not None and per_mile_cost > 0:
        def calculate_distance_cost(row):
            """Estimate cost to reach this restaurant from HQ (round-trip)"""
            if pd.isna(row.get('NODE_ID')):
                return float('inf')  # Unreachable — don't allocate
            
            node_id = int(row['NODE_ID'])
            # Look up distance from HQ to this restaurant
            distance_to_restaurant = distance_cache.get((hq_node, node_id), float('inf'))
            
            if distance_to_restaurant == float('inf'):
                return float('inf')  # No path exists
            
            # Estimate round-trip distance (there and back)
            estimated_round_trip = distance_to_restaurant * 2
            estimated_miles = estimated_round_trip / METERS_PER_MILE
            distance_cost = estimated_miles * per_mile_cost
            
            return distance_cost
        
        df['DISTANCE_COST'] = df.apply(calculate_distance_cost, axis=1)
        df['ADJUSTED_PROFIT'] = df['NET_PROFIT'] - df['DISTANCE_COST']
        
        # Only include restaurants with positive adjusted profit
        df = df[df['ADJUSTED_PROFIT'] > 0]
        
        # Sort by adjusted profit descending
        df_sorted = df.sort_values('ADJUSTED_PROFIT', ascending=False)
        
        print(f"   📍 Distance-adjusted selection: {len(df)} restaurants with positive adjusted profit")
        
    else:
        # Fallback: no distance data, use raw profit
        df['ADJUSTED_PROFIT'] = df['NET_PROFIT']
        df_sorted = df.sort_values('NET_PROFIT', ascending=False)
        print(f"   📍 No distance cache provided; using raw profit for selection")

    supply_left = effective_cap
    rows        = []

    for _, row in df_sorted.iterrows():
        if supply_left <= 0:
            break

        # Fill as many as supply allows — may be a partial fill on the last restaurant
        qty         = min(row['ENTREES_REQUESTED'], supply_left)
        supply_left -= qty

        rows.append({
            'RESTAURANT'        : row['RESTAURANT'],
            'ADDRESS'           : row['ADDRESS'],
            'LATITUDE'          : row['LATITUDE'],
            'LONGITUDE'         : row['LONGITUDE'],
            'NODE_ID'           : row['NODE_ID'],
            'ENTREES_ALLOCATED' : qty,
            'ENTREE_BID_PRICE'  : row['ENTREE_BID_PRICE'],
            'NET_MARGIN'        : row['NET_MARGIN'],
            'STOP_REVENUE'      : qty * row['ENTREE_BID_PRICE'],
            'STOP_SUPPLY_COST'  : qty * SUPPLY_COST_PER_ENTREE,
            'STOP_GROSS_PROFIT' : qty * row['NET_MARGIN'],
        })

    return pd.DataFrame(rows).reset_index(drop=True)

def build_route(alloc_df, G, hq_node, distance_cache=None):
    """Build a delivery route starting and ending at HQ (greedy nearest-neighbor TSP).

    Parameters
    ----------
    alloc_df : DataFrame from allocate_supply() — must have NODE_ID column
    G        : OSMnx directed road network graph
    hq_node  : int — graph node for the company HQ (start and end of route)
    distance_cache : dict, optional — pre-computed shortest paths for O(1) lookup
                     If provided, uses cache instead of calling nx.shortest_path_length()

    Returns
    -------
    DataFrame — stops in visitation order, with DISTANCE_FROM_PREV_M and
                CUMULATIVE_MILES. Distance from HQ to first stop and from
                last stop back to HQ is included in total miles.
    """
    if alloc_df.empty:
        return pd.DataFrame()   # nothing to route — return empty result

    def get_distance(from_node, to_node, G, cache):
        """Get distance using cache if available, otherwise compute."""
        if cache is not None:
            key = (from_node, to_node)
            if key not in cache:
                cache[key] = nx.shortest_path_length(G, from_node, to_node, weight='length')
            return cache[key]
        else:
            return nx.shortest_path_length(G, from_node, to_node, weight='length')

    route_rows   = []
    start        = alloc_df.iloc[0]

    # Distance from HQ to first restaurant
    dist_hq_to_first = get_distance(hq_node, int(start['NODE_ID']), G, distance_cache)
    route_rows.append({**start.to_dict(), 'DISTANCE_FROM_PREV_M': round(dist_hq_to_first, 1)})

    remaining    = alloc_df.iloc[1:].copy()
    current_node = int(start['NODE_ID'])

    while not remaining.empty:
        # Find the nearest unvisited restaurant by real road distance
        nearest_idx = remaining['NODE_ID'].apply(
            lambda n: get_distance(current_node, int(n), G, distance_cache)
        ).idxmin()

        nearest      = remaining.loc[nearest_idx]
        dist_m       = get_distance(current_node, int(nearest['NODE_ID']), G, distance_cache)

        route_rows.append({**nearest.to_dict(), 'DISTANCE_FROM_PREV_M': round(dist_m, 1)})

        remaining    = remaining.drop(nearest_idx)
        current_node = int(nearest['NODE_ID'])

    # Return leg: from last stop back to HQ
    dist_last_to_hq = get_distance(current_node, hq_node, G, distance_cache)
    route_rows.append({
        'RESTAURANT': 'HQ_RETURN',
        'ADDRESS': 'Return to HQ',
        'LATITUDE': None,
        'LONGITUDE': None,
        'NODE_ID': hq_node,
        'ENTREES_ALLOCATED': 0,
        'ENTREE_BID_PRICE': 0,
        'NET_MARGIN': 0,
        'STOP_REVENUE': 0,
        'STOP_SUPPLY_COST': 0,
        'STOP_GROSS_PROFIT': 0,
        'DISTANCE_FROM_PREV_M': round(dist_last_to_hq, 1),
    })

    route_df = pd.DataFrame(route_rows).reset_index(drop=True)

    # Ensure numeric dtypes after incremental concat
    for col in ['ENTREES_ALLOCATED', 'STOP_REVENUE', 'STOP_SUPPLY_COST',
                'STOP_GROSS_PROFIT', 'DISTANCE_FROM_PREV_M']:
        route_df[col] = pd.to_numeric(route_df[col], errors='coerce')

    route_df['CUMULATIVE_MILES'] = (
        route_df['DISTANCE_FROM_PREV_M'].cumsum() / METERS_PER_MILE
    ).round(2)

    return route_df

def calculate_pnl(route_df, truck):
    """Calculate the full P&L for a route + truck combination.
    Reads truck['fixed_cost'] and truck['per_mile_cost'].
    Returns a dict with revenue, supply_cost, gross_profit,
    truck_fixed_cost, truck_variable_cost, net_profit, entrees, miles.
    """
    # TODO: paste from X10D1
    if route_df.empty:
        return {
            'truck_name'        : truck['truck_name'],
            'stops'             : 0,
            'entrees'           : 0,
            'miles'             : 0,
            'revenue'           : 0,
            'supply_cost'       : 0,
            'gross_profit'      : 0,
            'truck_fixed_cost'  : 0,
            'truck_variable_cost': 0,
            'truck_cost'        : 0,
            'net_profit'        : 0,
        }

    total_miles    = route_df['DISTANCE_FROM_PREV_M'].sum() / METERS_PER_MILE
    revenue        = float(route_df['STOP_REVENUE'].sum())
    supply_cost    = float(route_df['STOP_SUPPLY_COST'].sum())
    gross_profit   = float(route_df['STOP_GROSS_PROFIT'].sum())
    truck_fixed    = truck['fixed_cost']
    truck_variable = total_miles * truck['per_mile_cost']
    truck_cost     = truck_fixed + truck_variable
    net_profit     = revenue - supply_cost - truck_cost

    return {
        'truck_name'        : truck['truck_name'],
        'stops'             : len(route_df),
        'entrees'           : int(route_df['ENTREES_ALLOCATED'].sum()),
        'miles'             : round(total_miles, 2),
        'revenue'           : round(revenue, 2),
        'supply_cost'       : round(supply_cost, 2),
        'gross_profit'      : round(gross_profit, 2),
        'truck_fixed_cost'  : round(truck_fixed, 2),
        'truck_variable_cost': round(truck_variable, 2),
        'truck_cost'        : round(truck_cost, 2),
        'net_profit'        : round(net_profit, 2),
    }

print('Functions defined.')

Functions defined.


## Phase 4: Distance Caching for Performance

Pre-compute all shortest paths between the HQ and all restaurants to enable O(1) lookups during optimization. This reduces runtime from 15+ minutes to ~30 seconds by eliminating repeated graph queries.

In [581]:
# ── Pre-compute ALL shortest paths for speed ─────────────────────────────────
# Build a cache of all node-to-node distances to avoid repeated graph queries
# This reduces 15 min runtime to ~30 seconds by turning O(log n) lookups into O(1)

import time
start_cache = time.time()

# Get all unique nodes: HQ + all geocoded restaurants
all_node_ids = set([HQ_NODE])
for _, row in df_raw.iterrows():
    if pd.notna(row.get('NODE_ID')):
        all_node_ids.add(int(row['NODE_ID']))

all_node_ids = sorted(list(all_node_ids))  # Convert to sorted list for consistency
print(f'Pre-computing distances between {len(all_node_ids)} nodes...')

# Build distance cache: (from_node, to_node) → distance_in_meters
distance_cache = {}
total_pairs = len(all_node_ids) ** 2
processed = 0

for i, from_node in enumerate(all_node_ids):
    for to_node in all_node_ids:
        if from_node != to_node:  # Skip same-node pairs
            try:
                dist = nx.shortest_path_length(G, from_node, to_node, weight='length')
                distance_cache[(from_node, to_node)] = dist
            except (nx.NetworkXNoPath, nx.NodeNotFound):
                # If no path exists, use large distance (won't be selected)
                distance_cache[(from_node, to_node)] = float('inf')
        
        processed += 1
        if processed % 500 == 0:
            print(f'  {processed}/{total_pairs} pairs computed ({100*processed/total_pairs:.0f}%)', end='\r')

elapsed_cache = time.time() - start_cache
cache_size = len(distance_cache)
print(f'\n✅ Cache built: {cache_size} distances pre-computed in {elapsed_cache:.1f}s')
print(f'   Memory usage: ~{cache_size * 56 / 1e6:.1f} MB')

Pre-computing distances between 32 nodes...
  1000/1024 pairs computed (98%)
✅ Cache built: 992 distances pre-computed in 94.9s
   Memory usage: ~0.1 MB


## Phase 5: Single-Truck Optimization

Test all 5 truck options as solo deployments. Establish baseline profitability and identify which truck generates the highest net profit when operating alone.

In [582]:
# ── Run all 5 single-truck scenarios ──────────────────────────────────────────
single_results = []

for truck in truck_list:
    alloc = allocate_supply(df_raw, DAILY_SUPPLY_CAP, truck['capacity'], 
                           hq_node=HQ_NODE, distance_cache=distance_cache, 
                           per_mile_cost=truck['per_mile_cost'])
    route = build_route(alloc, G, HQ_NODE, distance_cache)
    pnl   = calculate_pnl(route, truck)
    single_results.append({
        'truck':        truck['truck_name'],
        'capacity':     truck['capacity'],
        'fixed_cost':   truck['fixed_cost'],
        'per_mile':     truck['per_mile_cost'],
        'entrees':      pnl['entrees'],
        'miles':        pnl['miles'],
        'revenue':      pnl['revenue'],
        'supply_cost':  pnl['supply_cost'],
        'truck_cost':   pnl['truck_cost'],
        'net_profit':   pnl['net_profit'],
    })

single_df = pd.DataFrame(single_results).sort_values('net_profit', ascending=False)
single_df['total_cost'] = single_df['supply_cost'] + single_df['truck_cost']
print('=== Single-Truck Comparison ===')
display(single_df)
print(f'\nBest single-truck option: {single_df.iloc[0]["truck"]} '
      f'(${single_df.iloc[0]["net_profit"]:,.2f})')

   📍 Distance-adjusted selection: 27 restaurants with positive adjusted profit
   📍 Distance-adjusted selection: 27 restaurants with positive adjusted profit
   📍 Distance-adjusted selection: 28 restaurants with positive adjusted profit
   📍 Distance-adjusted selection: 28 restaurants with positive adjusted profit
   📍 Distance-adjusted selection: 27 restaurants with positive adjusted profit
=== Single-Truck Comparison ===


,truck,capacity,fixed_cost,per_mile,entrees,miles,revenue,supply_cost,truck_cost,net_profit,total_cost
4,Condor,999999,185,2.75,427,78.06,31460.0,22204.0,399.67,8856.33,22603.67
3,Albatross,220,240,1.25,220,63.24,17737.0,11440.0,319.05,5977.95,11759.05
2,Pelican,150,160,1.50,150,40.28,12454.0,7800.0,220.42,4433.58,8020.42
1,Heron,120,120,2.25,120,32.20,9862.0,6240.0,192.44,3429.56,6432.44
0,Kite,75,55,3.50,75,31.45,6154.0,3900.0,165.08,2088.92,4065.08



Best single-truck option: Condor ($8,856.33)


## Phase 6: Two-Truck Strategy 1 — Geographic (Latitude) Split

Divide restaurants into two groups using a simple geographic rule: line drawn at the median latitude (roughly separating north and south Dallas). This is a straightforward approach that respects natural geographic clusters.

In [583]:
# ── Two-truck split — design your strategy here ────────────────────────────────
# Example: split by latitude (north Dallas / south Dallas)
# You are free to use any split strategy — just document it.

# TODO: define df_truck1 and df_truck2 as subsets of df_raw
# Rules:
#   - No restaurant appears in both subsets
#   - Each subset has at least 3 restaurants
#   - Combined entrees allocated across both trucks <= DAILY_SUPPLY_CAP



# Example split by latitude:
median_lat = df_raw['LATITUDE'].median()
df_north   = df_raw[df_raw['LATITUDE'] >= median_lat].copy()
df_south   = df_raw[df_raw['LATITUDE'] <  median_lat].copy()

print(f'North restaurants: {len(df_north)}')
print(f'South restaurants: {len(df_south)}')
print('\n--- This is just a starting point ---')
print('Your group should experiment with different splits.')


North restaurants: 16
South restaurants: 16

--- This is just a starting point ---
Your group should experiment with different splits.


## Section 4 — Validation

Add at least **5 assert statements** validating key constraints and math.
Your notebook should be able to run top-to-bottom with all assertions passing.


## Phase 7: Route Mapping & Real Driving Times

Create interactive Folium maps showing the recommended single-truck and two-truck routes with actual driving paths. Integrate Open Route Service API to calculate real-world driving times instead of estimates.

In [584]:
# ── Open Route Service API: Real Driving Time Calculation ────────────────────
# Use ORS to calculate actual driving times for the best single and two-truck options
import requests
import json

# Calculate estimated driving times from routes (AVERAGE_SPEED_MPH from constants)
AVERAGE_SPEED_MPH = 30  # Standard average speed for urban delivery
best_single_driving_time = (best_single_route['DISTANCE_FROM_PREV_M'].sum() / METERS_PER_MILE) / AVERAGE_SPEED_MPH * 60
best_two_truck1_driving_time = (best_two_route1['DISTANCE_FROM_PREV_M'].sum() / METERS_PER_MILE) / AVERAGE_SPEED_MPH * 60
best_two_truck2_driving_time = (best_two_route2['DISTANCE_FROM_PREV_M'].sum() / METERS_PER_MILE) / AVERAGE_SPEED_MPH * 60

# Extract truck names from best two-truck result
best_two_truck1 = best_two['truck1']
best_two_truck2 = best_two['truck2']

# ⚠️  PASTE YOUR ORS API KEY HERE
ORS_API_KEY = 'eyJvcmciOiI1YjNjZTM1OTc4NTExMTAwMDFjZjYyNDgiLCJpZCI6ImYzYzVlZGIyMGE2MTQzZmI5NGVjODdlYjQ1OTNjZmNhIiwiaCI6Im11cm11cjY0In0='  # Get free key at https://openrouteservice.org/

def get_ors_driving_time(coordinates, api_key):
    """
    Calculate driving time using Open Route Service API.
    
    Parameters:
    -----------
    coordinates : list of [lon, lat] pairs (ordered waypoints)
    api_key : str, ORS API key
    
    Returns:
    --------
    dict with 'duration_seconds', 'duration_minutes', 'distance_m', 'distance_miles'
    or error dict if request fails
    """
    url = 'https://api.openrouteservice.org/v2/directions/driving-car'
    
    headers = {
        'Accept': 'application/json, application/geo+json',
        'Authorization': api_key,
        'Content-Type': 'application/json'
    }
    
    body = {'coordinates': coordinates}
    
    try:
        response = requests.post(url, json=body, headers=headers, timeout=30)
        response.raise_for_status()
        
        data = response.json()
        
        if 'routes' in data and len(data['routes']) > 0:
            route = data['routes'][0]
            summary = route.get('summary', {})
            
            duration_s = summary.get('duration', 0)
            distance_m = summary.get('distance', 0)
            
            return {
                'status': 'success',
                'duration_seconds': duration_s,
                'duration_minutes': round(duration_s / 60, 1),
                'distance_m': distance_m,
                'distance_miles': round(distance_m / METERS_PER_MILE, 2),
            }
        else:
            return {'status': 'error', 'message': 'No route found'}
    
    except requests.exceptions.RequestException as e:
        return {'status': 'error', 'message': str(e)}

def extract_route_coordinates(route_df):
    """
    Extract ordered [lon, lat] pairs from a route dataframe.
    Starts and ends at HQ.
    """
    coords = []
    
    # Start at HQ
    coords.append([HQ_LON, HQ_LAT])
    
    # Add all restaurant stops (exclude HQ_RETURN)
    stops = route_df[route_df['RESTAURANT'] != 'HQ_RETURN'].copy()
    for _, row in stops.iterrows():
        if pd.notna(row['LATITUDE']) and pd.notna(row['LONGITUDE']):
            coords.append([row['LONGITUDE'], row['LATITUDE']])
    
    # Return to HQ
    coords.append([HQ_LON, HQ_LAT])
    
    return coords

# Only run if API key is provided
if ORS_API_KEY != 'YOUR_API_KEY_HERE':
    print("=" * 80)
    print("🔍 CALCULATING REAL DRIVING TIMES WITH OPEN ROUTE SERVICE API")
    print("=" * 80)
    
    # Get coordinates for best single-truck route
    print(f"\n📍 Single-truck route ({best_single_truck})...")
    single_coords = extract_route_coordinates(best_single_route)
    print(f"   Route has {len(single_coords)-2} restaurant stops (plus HQ start/end)")
    
    single_ors_result = get_ors_driving_time(single_coords, ORS_API_KEY)
    
    if single_ors_result['status'] == 'success':
        ors_single_driving_time = single_ors_result['duration_minutes']
        ors_single_distance = single_ors_result['distance_miles']
        print(f"   ✅ ORS Driving time: {ors_single_driving_time:.1f} minutes ({ors_single_driving_time/60:.2f} hours)")
        print(f"   ✅ ORS Distance:     {ors_single_distance:.2f} miles")
        print(f"   📊 Estimate was:     {best_single_driving_time:.1f} minutes (@ {AVERAGE_SPEED_MPH} mph)")
    else:
        print(f"   ⚠️  Error: {single_ors_result.get('message', 'Unknown error')}")
        ors_single_driving_time = None
        ors_single_distance = None
    
    # Get coordinates for best two-truck routes
    print(f"\n📍 Two-truck route 1 ({best_two_truck1})...")
    truck1_coords = extract_route_coordinates(best_two_route1)
    print(f"   Route has {len(truck1_coords)-2} restaurant stops (plus HQ start/end)")
    
    truck1_ors_result = get_ors_driving_time(truck1_coords, ORS_API_KEY)
    
    if truck1_ors_result['status'] == 'success':
        ors_truck1_driving_time = truck1_ors_result['duration_minutes']
        ors_truck1_distance = truck1_ors_result['distance_miles']
        print(f"   ✅ ORS Driving time: {ors_truck1_driving_time:.1f} minutes ({ors_truck1_driving_time/60:.2f} hours)")
        print(f"   ✅ ORS Distance:     {ors_truck1_distance:.2f} miles")
        print(f"   📊 Estimate was:     {best_two_truck1_driving_time:.1f} minutes")
    else:
        print(f"   ⚠️  Error: {truck1_ors_result.get('message', 'Unknown error')}")
        ors_truck1_driving_time = None
        ors_truck1_distance = None
    
    print(f"\n📍 Two-truck route 2 ({best_two_truck2})...")
    truck2_coords = extract_route_coordinates(best_two_route2)
    print(f"   Route has {len(truck2_coords)-2} restaurant stops (plus HQ start/end)")
    
    truck2_ors_result = get_ors_driving_time(truck2_coords, ORS_API_KEY)
    
    if truck2_ors_result['status'] == 'success':
        ors_truck2_driving_time = truck2_ors_result['duration_minutes']
        ors_truck2_distance = truck2_ors_result['distance_miles']
        print(f"   ✅ ORS Driving time: {ors_truck2_driving_time:.1f} minutes ({ors_truck2_driving_time/60:.2f} hours)")
        print(f"   ✅ ORS Distance:     {ors_truck2_distance:.2f} miles")
        print(f"   📊 Estimate was:     {best_two_truck2_driving_time:.1f} minutes")
    else:
        print(f"   ⚠️  Error: {truck2_ors_result.get('message', 'Unknown error')}")
        ors_truck2_driving_time = None
        ors_truck2_distance = None
    
    # Combined two-truck time
    if ors_truck1_driving_time is not None and ors_truck2_driving_time is not None:
        ors_two_truck_total_time = ors_truck1_driving_time + ors_truck2_driving_time
        ors_two_truck_total_distance = ors_truck1_distance + ors_truck2_distance
    else:
        ors_two_truck_total_time = None
        ors_two_truck_total_distance = None
    
    print("\n" + "=" * 80)

else:
    print("⚠️  Open Route Service API key not configured.")
    print("   To use real driving time calculations:")
    print("   1. Get a free API key at https://openrouteservice.org/")
    print("   2. Replace 'YOUR_API_KEY_HERE' at the top of this cell")
    print("   3. Re-run this cell")
    print("\n   For now, using estimated times based on " + str(AVERAGE_SPEED_MPH) + " mph average speed.")
    
    ors_single_driving_time = None
    ors_single_distance = None
    ors_truck1_driving_time = None
    ors_truck1_distance = None
    ors_truck2_driving_time = None
    ors_truck2_distance = None
    ors_two_truck_total_time = None
    ors_two_truck_total_distance = None


🔍 CALCULATING REAL DRIVING TIMES WITH OPEN ROUTE SERVICE API

📍 Single-truck route (Condor)...
   Route has 25 restaurant stops (plus HQ start/end)
   ✅ ORS Driving time: 193.7 minutes (3.23 hours)
   ✅ ORS Distance:     98.51 miles
   📊 Estimate was:     156.1 minutes (@ 30 mph)

📍 Two-truck route 1 (Albatross)...
   Route has 12 restaurant stops (plus HQ start/end)
   ✅ ORS Driving time: 149.7 minutes (2.49 hours)
   ✅ ORS Distance:     82.06 miles
   📊 Estimate was:     128.0 minutes

📍 Two-truck route 2 (Condor)...
   Route has 13 restaurant stops (plus HQ start/end)
   ✅ ORS Driving time: 86.6 minutes (1.44 hours)
   ✅ ORS Distance:     36.01 miles
   📊 Estimate was:     62.2 minutes



In [585]:
# ── Detailed Summary Analysis ────────────────────────────────────────────────
# Create a comprehensive comparison showing revenue, costs, distance, and driving time
# Uses ORS real driving times if available, otherwise uses estimated times

print("=" * 80)
print("DETAILED FINANCIAL & OPERATIONAL SUMMARY")
print("=" * 80)

# Get best single-truck details
best_single_truck_name = single_df.iloc[0]['truck']
best_single_revenue = single_df.iloc[0]['revenue']
best_single_supply_cost = single_df.iloc[0]['supply_cost']
best_single_truck_cost = single_df.iloc[0]['truck_cost']
best_single_net_profit = single_df.iloc[0]['net_profit']
best_single_miles = single_df.iloc[0]['miles']

# Use ORS distance if available, otherwise use graph distance
if ors_single_distance is not None:
    display_single_miles = ors_single_distance
    miles_source_single = "(ORS API)"
else:
    display_single_miles = best_single_miles
    miles_source_single = "(OSMnx graph)"

# Use ORS driving time if available, otherwise use estimate
if ors_single_driving_time is not None:
    display_single_driving_time = ors_single_driving_time
    time_source_single = "(ORS API)"
else:
    display_single_driving_time = best_single_driving_time
    time_source_single = f"(estimated @ {AVERAGE_SPEED_MPH} mph)"

best_single_entrees = int(single_df.iloc[0]['entrees'])

# Get best two-truck details
best_two_truck1 = best_two['truck1']
best_two_truck2 = best_two['truck2']
best_two_truck1_revenue = best_two['truck1_profit'] + best_two['truck1_entrees'] * SUPPLY_COST_PER_ENTREE
best_two_truck2_revenue = best_two['truck2_profit'] + best_two['truck2_entrees'] * SUPPLY_COST_PER_ENTREE
best_two_total_revenue = best_two_truck1_revenue + best_two_truck2_revenue
best_two_truck1_supply = best_two['truck1_entrees'] * SUPPLY_COST_PER_ENTREE
best_two_truck2_supply = best_two['truck2_entrees'] * SUPPLY_COST_PER_ENTREE
best_two_total_supply = best_two_truck1_supply + best_two_truck2_supply
best_two_total_miles = best_two['truck1_miles'] + best_two['truck2_miles']

# Use ORS times if available for truck 1
if ors_truck1_driving_time is not None:
    display_truck1_driving_time = ors_truck1_driving_time
    display_truck1_miles = ors_truck1_distance
    time_source_truck1 = "(ORS API)"
else:
    display_truck1_driving_time = best_two_truck1_driving_time
    display_truck1_miles = best_two['truck1_miles']
    time_source_truck1 = f"(estimated @ {AVERAGE_SPEED_MPH} mph)"

# Use ORS times if available for truck 2
if ors_truck2_driving_time is not None:
    display_truck2_driving_time = ors_truck2_driving_time
    display_truck2_miles = ors_truck2_distance
    time_source_truck2 = "(ORS API)"
else:
    display_truck2_driving_time = best_two_truck2_driving_time
    display_truck2_miles = best_two['truck2_miles']
    time_source_truck2 = f"(estimated @ {AVERAGE_SPEED_MPH} mph)"

# Use ORS total if available
if ors_two_truck_total_time is not None:
    display_two_truck_total_time = ors_two_truck_total_time
    display_two_truck_total_miles = ors_two_truck_total_distance
    time_source_two_total = "(ORS API)"
else:
    display_two_truck_total_time = best_two_truck1_driving_time + best_two_truck2_driving_time
    display_two_truck_total_miles = best_two_total_miles

# Calculate truck costs using the same formula as single-truck scenario
# Truck Cost = Fixed Cost + (Miles * Per-Mile Cost)
truck1_dict = next(t for t in truck_list if t['truck_name'] == best_two_truck1)
truck2_dict = next(t for t in truck_list if t['truck_name'] == best_two_truck2)

best_two_truck1_cost = truck1_dict['fixed_cost'] + (display_truck1_miles * truck1_dict['per_mile_cost'])
best_two_truck2_cost = truck2_dict['fixed_cost'] + (display_truck2_miles * truck2_dict['per_mile_cost'])
best_two_total_truck_cost = best_two_truck1_cost + best_two_truck2_cost

best_two_total_entrees = int(best_two['truck1_entrees'] + best_two['truck2_entrees'])
best_two_total_profit = best_two['combined_profit']

print("\n📦 BEST SINGLE-TRUCK OPTION: " + best_single_truck_name.upper())
print("-" * 80)
print(f"  Restaurants served:        {len(single_alloc)} stops")
print(f"  Total entrees delivered:   {best_single_entrees} entrees")
print(f"  Total distance:            {display_single_miles:.2f} miles {miles_source_single}")
print(f"  Driving time:              {display_single_driving_time:.1f} minutes ({display_single_driving_time/60:.2f} hours) {time_source_single}")
print(f"\n  Revenue:                   ${best_single_revenue:>12,.2f}")
print(f"  Supply cost:               ${best_single_supply_cost:>12,.2f}")
print(f"  Truck cost (fixed+var):    ${best_single_truck_cost:>12,.2f}")
print(f"  ─" * 40)
print(f"  NET PROFIT:                ${best_single_net_profit:>12,.2f}")

print("\n\n🚚 BEST TWO-TRUCK OPTION: " + best_two_truck1.upper() + " + " + best_two_truck2.upper())
print("-" * 80)
print(f"  {best_two_truck1}:")
print(f"    - Restaurants served:    {len(two_alloc1)} stops")
print(f"    - Entrees delivered:     {int(best_two['truck1_entrees'])} entrees")
print(f"    - Distance:              {display_truck1_miles:.2f} miles {time_source_truck1}")
print(f"    - Driving time:          {display_truck1_driving_time:.1f} minutes ({display_truck1_driving_time/60:.2f} hours) {time_source_truck1}")
print(f"    - Revenue:               ${best_two_truck1_revenue:>12,.2f}")
print(f"    - Supply cost:           ${best_two_truck1_supply:>12,.2f}")
print(f"    - Truck cost:            ${best_two_truck1_cost:>12,.2f}")
print(f"    - Net profit:            ${best_two['truck1_profit']:>12,.2f}")

print(f"\n  {best_two_truck2}:")
print(f"    - Restaurants served:    {len(two_alloc2)} stops")
print(f"    - Entrees delivered:     {int(best_two['truck2_entrees'])} entrees")
print(f"    - Distance:              {display_truck2_miles:.2f} miles {time_source_truck2}")
print(f"    - Driving time:          {display_truck2_driving_time:.1f} minutes ({display_truck2_driving_time/60:.2f} hours) {time_source_truck2}")
print(f"    - Revenue:               ${best_two_truck2_revenue:>12,.2f}")
print(f"    - Supply cost:           ${best_two_truck2_supply:>12,.2f}")
print(f"    - Truck cost:            ${best_two_truck2_cost:>12,.2f}")
print(f"    - Net profit:            ${best_two['truck2_profit']:>12,.2f}")

print(f"\n  COMBINED TOTALS:")
print(f"    - Total restaurants:     {len(two_alloc1) + len(two_alloc2)} stops")
print(f"    - Total entrees:         {best_two_total_entrees} entrees")
print(f"    - Total distance:        {display_two_truck_total_miles:.2f} miles {time_source_two_total}")
print(f"    - Total driving time:    {display_two_truck_total_time:.1f} minutes ({display_two_truck_total_time/60:.2f} hours) {time_source_two_total}")
print(f"    - Total revenue:         ${best_two_total_revenue:>12,.2f}")
print(f"    - Total supply cost:     ${best_two_total_supply:>12,.2f}")
print(f"    - Total truck cost:      ${best_two_total_truck_cost:>12,.2f}")
print(f"    - ─" * 40)
print(f"    - NET PROFIT:            ${best_two_total_profit:>12,.2f}")

print("\n\n📊 HEAD-TO-HEAD COMPARISON")
print("-" * 80)
profit_diff = best_two_total_profit - best_single_net_profit
profit_pct = (profit_diff / best_single_net_profit * 100) if best_single_net_profit > 0 else 0
time_diff = display_two_truck_total_time - display_single_driving_time

print(f"  Single-truck net profit:   ${best_single_net_profit:>12,.2f}")
print(f"  Two-truck net profit:      ${best_two_total_profit:>12,.2f}")
print(f"  Difference:                ${profit_diff:>12,.2f} ({profit_pct:+.1f}%)")

print(f"\n  Single-truck driving time: {display_single_driving_time:>12.1f} minutes")
print(f"  Two-truck driving time:    {display_two_truck_total_time:>12.1f} minutes")
print(f"  Difference:                {time_diff:>12.1f} minutes ({time_diff/60:+.2f} hours)")

if best_two_total_profit > best_single_net_profit:
    print(f"\n  ✅ TWO-TRUCK OPTION IS MORE PROFITABLE!")
    print(f"     Two trucks generate ${profit_diff:,.2f} more profit.")
else:
    print(f"\n  ✅ SINGLE-TRUCK OPTION IS MORE PROFITABLE!")
    print(f"     One truck saves ${abs(profit_diff):,.2f} in profit loss.")

# Efficiency metrics
single_revenue_per_mile = best_single_revenue / display_single_miles if display_single_miles > 0 else 0
two_revenue_per_mile = best_two_total_revenue / display_two_truck_total_miles if display_two_truck_total_miles > 0 else 0

print(f"\n  Efficiency (Revenue per mile):")
print(f"    Single-truck:            ${single_revenue_per_mile:>12,.2f}/mile")
print(f"    Two-truck:               ${two_revenue_per_mile:>12,.2f}/mile")

single_revenue_per_entree = best_single_revenue / best_single_entrees if best_single_entrees > 0 else 0
two_revenue_per_entree = best_two_total_revenue / best_two_total_entrees if best_two_total_entrees > 0 else 0

print(f"\n  Revenue per entree delivered:")
print(f"    Single-truck:            ${single_revenue_per_entree:>12,.2f}/entree")
print(f"    Two-truck:               ${two_revenue_per_entree:>12,.2f}/entree")

print(f"\n  Profit per hour of driving time:")
single_profit_per_hour = best_single_net_profit / (display_single_driving_time / 60) if display_single_driving_time > 0 else 0
two_profit_per_hour = best_two_total_profit / (display_two_truck_total_time / 60) if display_two_truck_total_time > 0 else 0
print(f"    Single-truck:            ${single_profit_per_hour:>12,.2f}/hour")
print(f"    Two-truck:               ${two_profit_per_hour:>12,.2f}/hour")

print("\n" + "=" * 80)


DETAILED FINANCIAL & OPERATIONAL SUMMARY

📦 BEST SINGLE-TRUCK OPTION: CONDOR
--------------------------------------------------------------------------------
  Restaurants served:        25 stops
  Total entrees delivered:   427 entrees
  Total distance:            98.51 miles (ORS API)
  Driving time:              193.7 minutes (3.23 hours) (ORS API)

  Revenue:                   $   31,460.00
  Supply cost:               $   22,204.00
  Truck cost (fixed+var):    $      399.67
  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─
  NET PROFIT:                $    8,856.33


🚚 BEST TWO-TRUCK OPTION: ALBATROSS + CONDOR
--------------------------------------------------------------------------------
  Albatross:
    - Restaurants served:    12 stops
    - Entrees delivered:     220 entrees
    - Distance:              82.06 miles (ORS API)
    - Driving time:          149.7 minutes (2.49 hours) (ORS API)
    - Revenue:  

## Phase 8: Demand Concentration Analysis

Create a heat map visualizing which restaurants generate the most demand (entrees requested). Shows which areas of Dallas have the highest profit concentration and how well the recommended routes serve top-demand restaurants.

## Section 2.6 — Geographic Clustering for Two-Truck Split

So far, we've split restaurants between trucks by drawing a line across the middle of Dallas
(north vs south). That works, but it doesn't account for the actual road network. Two restaurants
on opposite sides of the latitude line might be very close to each other on the highway, while
two restaurants on the same side of the line might be far apart.

**K-means clustering** is a smarter approach. Instead of drawing an arbitrary geographic line,
we group restaurants into clusters based on how close they actually are to each other.
The algorithm finds the best way to split 29 restaurants into 2 groups so that each group
is geographically compact — restaurants within a cluster are near each other, and clusters
are far from each other.

Below, we:
1. Use k-means to find these 2 natural clusters based on latitude and longitude
2. Visualize them on a map so we can see if they make intuitive sense
3. Run the full two-truck optimization using the clustered split instead of the latitude split
4. Compare: does clustering beat the simple north/south division?

The goal: a more efficient routing that respects actual geography, not arbitrary boundaries.

## Phase 9: Two-Truck Strategy 2 — K-Means Clustering

Use k-means clustering to group restaurants based on actual proximity, not arbitrary latitude lines. This approach finds the optimal geographic split that minimizes distances within each cluster.

In [586]:
# ── K-Means Geographic Clustering ─────────────────────────────────────────
from sklearn.cluster import KMeans
import numpy as np

print("=" * 80)
print("🎯 K-MEANS GEOGRAPHIC CLUSTERING")
print("=" * 80)

# Prepare data for clustering: use restaurants with valid coordinates
df_for_clustering = df_raw.dropna(subset=['LATITUDE', 'LONGITUDE']).copy()

print(f"\n📍 Clustering {len(df_for_clustering)} restaurants by geographic location...")

# Extract coordinates and perform k-means clustering
coords = df_for_clustering[['LATITUDE', 'LONGITUDE']].values
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
df_for_clustering['CLUSTER'] = kmeans.fit_predict(coords)

# Write cluster labels back to main dataframe
df_raw.loc[df_for_clustering.index, 'CLUSTER'] = df_for_clustering['CLUSTER']

# Split into clusters
df_cluster0 = df_for_clustering[df_for_clustering['CLUSTER'] == 0].copy()
df_cluster1 = df_for_clustering[df_for_clustering['CLUSTER'] == 1].copy()

# Calculate cluster statistics
cluster0_count = len(df_cluster0)
cluster1_count = len(df_cluster1)
cluster0_demand = int(df_cluster0['ENTREES_REQUESTED'].sum())
cluster1_demand = int(df_cluster1['ENTREES_REQUESTED'].sum())

# Geographic centers of each cluster
cluster0_center_lat = df_cluster0['LATITUDE'].mean()
cluster0_center_lon = df_cluster0['LONGITUDE'].mean()
cluster1_center_lat = df_cluster1['LATITUDE'].mean()
cluster1_center_lon = df_cluster1['LONGITUDE'].mean()

print(f"\n✅ Cluster 0 (Blue):")
print(f"   Restaurants: {cluster0_count}")
print(f"   Total demand: {cluster0_demand} entrees")
print(f"   Geographic center: ({cluster0_center_lat:.4f}°N, {cluster0_center_lon:.4f}°W)")

print(f"\n✅ Cluster 1 (Red):")
print(f"   Restaurants: {cluster1_count}")
print(f"   Total demand: {cluster1_demand} entrees")
print(f"   Geographic center: ({cluster1_center_lat:.4f}°N, {cluster1_center_lon:.4f}°W)")

print(f"\n{'='*80}")

🎯 K-MEANS GEOGRAPHIC CLUSTERING

📍 Clustering 32 restaurants by geographic location...

✅ Cluster 0 (Blue):
   Restaurants: 23
   Total demand: 406 entrees
   Geographic center: (32.8065°N, -96.7876°W)

✅ Cluster 1 (Red):
   Restaurants: 9
   Total demand: 183 entrees
   Geographic center: (32.9169°N, -96.8417°W)



c:\Users\samdu\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1425: UserWarning:

KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.



In [587]:
# ── Cluster Visualization Map ─────────────────────────────────────────────
import folium
from folium.plugins import MarkerCluster

print("🗺️  Creating geographic cluster map...")

# Create map centered on Dallas
cluster_map = folium.Map(
    location=[HQ_LAT, HQ_LON],
    zoom_start=11,
    tiles='OpenStreetMap'
)

# Add Cluster 0 markers (blue)
for _, row in df_cluster0.iterrows():
    folium.CircleMarker(
        location=[row['LATITUDE'], row['LONGITUDE']],
        radius=7,
        popup=f"{row['RESTAURANT']}<br>{int(row['ENTREES_REQUESTED'])} entrees",
        color='blue',
        fill=True,
        fillColor='blue',
        fillOpacity=0.7,
        weight=2,
        opacity=0.9
    ).add_to(cluster_map)

# Add Cluster 1 markers (red)
for _, row in df_cluster1.iterrows():
    folium.CircleMarker(
        location=[row['LATITUDE'], row['LONGITUDE']],
        radius=7,
        popup=f"{row['RESTAURANT']}<br>{int(row['ENTREES_REQUESTED'])} entrees",
        color='red',
        fill=True,
        fillColor='red',
        fillOpacity=0.7,
        weight=2,
        opacity=0.9
    ).add_to(cluster_map)

# Add HQ marker
folium.Marker(
    location=[HQ_LAT, HQ_LON],
    popup='HQ - Headquarters',
    icon=folium.Icon(color='darkpurple', icon='home', prefix='fa'),
    tooltip='Headquarters'
).add_to(cluster_map)

# Add cluster centers as larger markers
folium.CircleMarker(
    location=[cluster0_center_lat, cluster0_center_lon],
    radius=10,
    popup='Cluster 0 Center',
    color='darkblue',
    fill=True,
    fillColor='lightblue',
    fillOpacity=0.9,
    weight=3
).add_to(cluster_map)

folium.CircleMarker(
    location=[cluster1_center_lat, cluster1_center_lon],
    radius=10,
    popup='Cluster 1 Center',
    color='darkred',
    fill=True,
    fillColor='lightcoral',
    fillOpacity=0.9,
    weight=3
).add_to(cluster_map)

# Add legend
legend_html = '''
<div style="position: fixed; 
     bottom: 50px; right: 50px; width: 280px; height: auto; 
     background-color: white; border:2px solid grey; z-index:9999; 
     font-size:12px; padding: 15px; border-radius: 5px; box-shadow: 2px 2px 6px rgba(0,0,0,0.3); font-family: Arial, sans-serif;">
     <b style="font-size: 14px;">📍 K-MEANS CLUSTERS</b><br>
     <hr style="margin: 8px 0; border: none; border-top: 1px solid #ccc;">
     <span style="display:inline-block; width:12px; height:12px; background:blue; border-radius:50%; margin-right:5px;"></span><b>Cluster 0 (Blue)</b><br>
     <span style="display:inline-block; width:12px; height:12px; background:red; border-radius:50%; margin-right:5px;"></span><b>Cluster 1 (Red)</b><br>
     <span style="display:inline-block; width:12px; height:12px; background:darkpurple; border-radius:50%; margin-right:5px;"></span><b>HQ</b><br>
     <br>
     <small>Larger circles = cluster geographic centers</small>
</div>
'''

cluster_map.get_root().html.add_child(folium.Element(legend_html))

# Save map
cluster_map.save('cluster_map.html')
print(f"✅ Cluster map saved to cluster_map.html")

🗺️  Creating geographic cluster map...
✅ Cluster map saved to cluster_map.html


In [588]:
# ── Two-Truck Pipeline with Clustering ────────────────────────────────────
from itertools import combinations
import time

print("\n" + "=" * 80)
print("🚚 TWO-TRUCK OPTIMIZATION WITH K-MEANS CLUSTERS")
print("=" * 80)

two_truck_cluster_results = []
start_time = time.time()

# Count total scenarios
total_truck_pairs = len(list(combinations(range(len(truck_list)), 2)))
total_cap_splits = len(list(range(30, DAILY_SUPPLY_CAP, 10)))
total_scenarios = total_truck_pairs * total_cap_splits

print(f"\nTesting {total_truck_pairs} truck combinations × {total_cap_splits} cap splits = {total_scenarios} scenarios")
print('Progress: ', end='', flush=True)

scenario_count = 0

# Test all combinations of 2 different trucks with clustered split
for truck_idx1, truck_idx2 in combinations(range(len(truck_list)), 2):
    truck1 = truck_list[truck_idx1]
    truck2 = truck_list[truck_idx2]
    
    # Test different cap splits (various allocations of DAILY_SUPPLY_CAP entrees)
    for cap1 in range(30, DAILY_SUPPLY_CAP, 10):
        cap2 = DAILY_SUPPLY_CAP - cap1
        
        # Use cluster-based split for restaurants
        alloc1 = allocate_supply(df_cluster0, cap1, truck1['capacity'])
        route1 = build_route(alloc1, G, HQ_NODE, distance_cache)
        pnl1   = calculate_pnl(route1, truck1)
        
        alloc2 = allocate_supply(df_cluster1, cap2, truck2['capacity'])
        route2 = build_route(alloc2, G, HQ_NODE, distance_cache)
        pnl2   = calculate_pnl(route2, truck2)
        
        combined_profit = pnl1['net_profit'] + pnl2['net_profit']
        
        two_truck_cluster_results.append({
            'truck1': truck1['truck_name'],
            'truck2': truck2['truck_name'],
            'cap_split': f'{cap1}/{cap2}',
            'truck1_profit': pnl1['net_profit'],
            'truck2_profit': pnl2['net_profit'],
            'combined_profit': combined_profit,
            'truck1_entrees': pnl1['entrees'],
            'truck2_entrees': pnl2['entrees'],
            'truck1_miles': pnl1['miles'],
            'truck2_miles': pnl2['miles'],
        })
        
        scenario_count += 1
        if scenario_count % 10 == 0:
            print(f'{scenario_count}', end=' ', flush=True)

print(f'\nCompleted {scenario_count} scenarios in {time.time()-start_time:.1f}s\n')

# Create comparison table
two_truck_cluster_df = pd.DataFrame(two_truck_cluster_results).sort_values('combined_profit', ascending=False)

print('=== Two-Truck Scenario Comparison (K-Means Clusters) ===')
display(two_truck_cluster_df.head(15))

best_cluster = two_truck_cluster_df.iloc[0]
print(f'\n✅ Best two-truck configuration (K-means clustering):')
print(f'   Trucks: {best_cluster["truck1"]} + {best_cluster["truck2"]}')
print(f'   Cap split: {best_cluster["cap_split"]} entrees')
print(f'   Combined net profit: ${best_cluster["combined_profit"]:,.2f}')
print(f'   {best_cluster["truck1"]}: ${best_cluster["truck1_profit"]:,.2f} ({int(best_cluster["truck1_entrees"])} entrees, {best_cluster["truck1_miles"]} miles)')
print(f'   {best_cluster["truck2"]}: ${best_cluster["truck2_profit"]:,.2f} ({int(best_cluster["truck2_entrees"])} entrees, {best_cluster["truck2_miles"]} miles)')


🚚 TWO-TRUCK OPTIMIZATION WITH K-MEANS CLUSTERS

Testing 10 truck combinations × 40 cap splits = 400 scenarios
Progress:    📍 No distance cache provided; using raw profit for selection
   📍 No distance cache provided; using raw profit for selection
   📍 No distance cache provided; using raw profit for selection
   📍 No distance cache provided; using raw profit for selection
   📍 No distance cache provided; using raw profit for selection
   📍 No distance cache provided; using raw profit for selection
   📍 No distance cache provided; using raw profit for selection
   📍 No distance cache provided; using raw profit for selection
   📍 No distance cache provided; using raw profit for selection
   📍 No distance cache provided; using raw profit for selection
   📍 No distance cache provided; using raw profit for selection
   📍 No distance cache provided; using raw profit for selection
   📍 No distance cache provided; using raw profit for selection
   📍 No distance cache provided; using raw prof

,truck1,truck2,cap_split,truck1_profit,truck2_profit,combined_profit,truck1_entrees,truck2_entrees,truck1_miles,truck2_miles
380,Albatross,Condor,230/197,5204.02,2860.66,8064.68,220,183,41.58,42.31
379,Albatross,Condor,220/207,5204.02,2860.66,8064.68,220,183,41.58,42.31
381,Albatross,Condor,240/187,5204.02,2860.66,8064.68,220,183,41.58,42.31
382,Albatross,Condor,250/177,5204.02,2860.66,8064.68,220,177,41.58,42.31
383,Albatross,Condor,260/167,5204.02,2860.66,8064.68,220,167,41.58,42.31
384,Albatross,Condor,270/157,5204.02,2860.66,8064.68,220,157,41.58,42.31
385,Albatross,Condor,280/147,5204.02,2828.66,8032.68,220,147,41.58,42.31
386,Albatross,Condor,290/137,5204.02,2788.66,7992.68,220,137,41.58,42.31
387,Albatross,Condor,300/127,5204.02,2752.69,7956.71,220,127,41.58,34.29
378,Albatross,Condor,210/217,5003.66,2860.66,7864.32,210,183,37.07,42.31



✅ Best two-truck configuration (K-means clustering):
   Trucks: Albatross + Condor
   Cap split: 230/197 entrees
   Combined net profit: $8,064.68
   Albatross: $5,204.02 (220 entrees, 41.58 miles)
   Condor: $2,860.66 (183 entrees, 42.31 miles)


In [589]:
# ── Clustering vs Latitude Split Comparison ───────────────────────────────
print("\n" + "=" * 80)
print("📊 K-MEANS CLUSTERING VS LATITUDE SPLIT COMPARISON")
print("=" * 80)

# Get best results from both methods
best_latitude = two_truck_df.iloc[0]
best_cluster = two_truck_cluster_df.iloc[0]

# Create comparison
comparison_data = {
    'Method': ['Latitude Split (North/South)', 'K-Means Clustering'],
    'Best Truck Pair': [f"{best_latitude['truck1']} + {best_latitude['truck2']}", 
                        f"{best_cluster['truck1']} + {best_cluster['truck2']}"],
    'Cap Split': [best_latitude['cap_split'], best_cluster['cap_split']],
    'Combined Profit': [f"${best_latitude['combined_profit']:,.2f}", 
                       f"${best_cluster['combined_profit']:,.2f}"],
    'Total Miles': [f"{best_latitude['truck1_miles'] + best_latitude['truck2_miles']:.1f}", 
                   f"{best_cluster['truck1_miles'] + best_cluster['truck2_miles']:.1f}"],
}

comparison_df = pd.DataFrame(comparison_data)
print()
display(comparison_df)

# Calculate improvement
profit_diff = best_cluster['combined_profit'] - best_latitude['combined_profit']
profit_pct = (profit_diff / abs(best_latitude['combined_profit']) * 100) if best_latitude['combined_profit'] != 0 else 0

miles_diff = (best_cluster['truck1_miles'] + best_cluster['truck2_miles']) - \
             (best_latitude['truck1_miles'] + best_latitude['truck2_miles'])

print(f"\n💡 TAKEAWAY:")
if profit_diff > 50:
    print(f"   ✅ K-means clustering wins! +${profit_diff:,.2f} profit (+{profit_pct:.1f}%) vs latitude split.")
    print(f"   Total miles: {miles_diff:+.1f} miles")
elif profit_diff < -50:
    print(f"   📊 Latitude split performs better: +${abs(profit_diff):,.2f} profit vs clustering.")
    print(f"   (Geographic clustering didn't improve on the natural north/south division.)")
else:
    print(f"   📊 Results are very similar (within ${abs(profit_diff):,.2f}).")
    print(f"   (The restaurants naturally cluster north/south, so latitude split is fine.)")

print(f"{'='*80}")


📊 K-MEANS CLUSTERING VS LATITUDE SPLIT COMPARISON



,Method,Best Truck Pair,Cap Split,Combined Profit,Total Miles
0,Latitude Split (North/South),Albatross + Condor,230/197,"$8,560.43",95.1
1,K-Means Clustering,Albatross + Condor,230/197,"$8,064.68",83.9



💡 TAKEAWAY:
   📊 Latitude split performs better: +$495.75 profit vs clustering.
   (Geographic clustering didn't improve on the natural north/south division.)


## Section 2.7: Demand-Based Split Strategy

Instead of using geography to assign restaurants to trucks, what if we balance the trucks by **demand volume**? This approach sorts restaurants by entree demand (descending) and alternates assignment, ensuring both trucks carry similar total demand loads. 

**Rationale:** Balanced demand allocation can minimize idle truck capacity and may reduce overall supply costs by improving utilization efficiency.

**Process:**
1. Sort all restaurants by ENTREES_REQUESTED (highest first)
2. Alternate assignment: odd-indexed restaurants → Truck 1, even-indexed → Truck 2
3. Run full two-truck optimization pipeline (all 10 truck pair combinations × cap splits)
4. Compare results against latitude split and k-means clustering

## Phase 10: Two-Truck Strategy 3 — Demand-Based Allocation

Balance truck loads by demand: sort restaurants by entree requests (highest first) and alternate assignment. This approach prioritizes balanced capacity utilization over pure geography.

In [590]:
# ── Demand-Based Split: Balanced Allocation ─────────────────────────────
# Sort by demand (descending) and alternate assignment

df_raw_sorted = df_raw.dropna(subset=['LATITUDE', 'LONGITUDE']).sort_values(
    'ENTREES_REQUESTED', ascending=False
).reset_index(drop=True)

# Alternate assignment: index 0,2,4... → demand0; index 1,3,5... → demand1
df_demand0 = df_raw_sorted[df_raw_sorted.index % 2 == 0].reset_index(drop=True)
df_demand1 = df_raw_sorted[df_raw_sorted.index % 2 == 1].reset_index(drop=True)

print(f"✓ Demand-Based Split Complete")
print(f"  Truck 1 (Demand Balanced): {len(df_demand0)} restaurants, "
      f"{df_demand0['ENTREES_REQUESTED'].sum()} total entrees")
print(f"  Truck 2 (Demand Balanced): {len(df_demand1)} restaurants, "
      f"{df_demand1['ENTREES_REQUESTED'].sum()} total entrees")
print(f"  Demand balance ratio: {df_demand0['ENTREES_REQUESTED'].sum() / df_demand1['ENTREES_REQUESTED'].sum():.3f}")
print(f"\n  Sample split:")
print(f"    Top 5 by demand: {df_raw_sorted['RESTAURANT'].head(5).tolist()}")
print(f"    Distribution: Truck 1={list(df_demand0['RESTAURANT'].head(3).values)}")
print(f"                  Truck 2={list(df_demand1['RESTAURANT'].head(3).values)}")

✓ Demand-Based Split Complete
  Truck 1 (Demand Balanced): 16 restaurants, 302 total entrees
  Truck 2 (Demand Balanced): 16 restaurants, 287 total entrees
  Demand balance ratio: 1.052

  Sample split:
    Top 5 by demand: ["Ruth's Chris Cuy House", 'Cuy-Fil-A', "Pappasito's Cuytina", 'Cuysteak House', 'Cuy de Brazil']
    Distribution: Truck 1=["Ruth's Chris Cuy House", "Pappasito's Cuytina", 'Cuy de Brazil']
                  Truck 2=['Cuy-Fil-A', 'Cuysteak House', "TD's Barbecuy"]


In [591]:
# ── Two-Truck Optimization with Demand Split ─────────────────────────────
import time

start_time_demand = time.time()
two_truck_demand_results = []
scenario_count_demand = 0
total_scenarios_demand = (len(truck_list) - 1) * (len(truck_list) - 2) // 2 * 6  # 10 pairs × 6 cap splits

for truck_idx1 in range(len(truck_list) - 1):
    for truck_idx2 in range(truck_idx1 + 1, len(truck_list)):
        truck1_dict = truck_list[truck_idx1]
        truck2_dict = truck_list[truck_idx2]
        
        for cap_split in range(30, DAILY_SUPPLY_CAP, 25):  # Capacity splits: 30, 55, 80, 105, 130, 155
            cap1, cap2 = cap_split, DAILY_SUPPLY_CAP - cap_split
            
            truck1_cap = truck1_dict['capacity']
            truck2_cap = truck2_dict['capacity']
            
            # Allocate to both trucks based on split
            alloc1 = allocate_supply(df_demand0, cap1, truck1_cap)
            alloc2 = allocate_supply(df_demand1, cap2, truck2_cap)
            
            allocated1 = alloc1['ENTREES_ALLOCATED'].sum() if not alloc1.empty else 0
            allocated2 = alloc2['ENTREES_ALLOCATED'].sum() if not alloc2.empty else 0
            unmet1 = max(0, df_demand0['ENTREES_REQUESTED'].sum() - allocated1)
            unmet2 = max(0, df_demand1['ENTREES_REQUESTED'].sum() - allocated2)
            
            # Build routes
            route1 = build_route(alloc1, G, HQ_NODE, distance_cache)
            route2 = build_route(alloc2, G, HQ_NODE, distance_cache)
            
            # Calculate P&L
            pnl1 = calculate_pnl(route1, truck1_dict)
            pnl2 = calculate_pnl(route2, truck2_dict)
            
            combined_pnl = pnl1['net_profit'] + pnl2['net_profit']
            
            two_truck_demand_results.append({
                'truck_pair': f"{truck1_dict['truck_name']}/{truck2_dict['truck_name']}",
                'truck1': truck1_dict['truck_name'],
                'truck2': truck2_dict['truck_name'],
                'cap_split': f"{cap1}/{cap2}",
                'total_entrees': allocated1 + allocated2,
                'total_revenue': pnl1['revenue'] + pnl2['revenue'],
                'supply_cost': pnl1['supply_cost'] + pnl2['supply_cost'],
                'truck_cost': pnl1['truck_cost'] + pnl2['truck_cost'],
                'travel_miles': pnl1['miles'] + pnl2['miles'],
                'net_profit': combined_pnl
            })
            
            scenario_count_demand += 1
            if scenario_count_demand % 10 == 0:
                print(f"  Scenario {scenario_count_demand}/{total_scenarios_demand}...", end='\r')

two_truck_demand_df = pd.DataFrame(two_truck_demand_results).sort_values('net_profit', ascending=False)
best_demand = two_truck_demand_df.iloc[0]

elapsed_demand = time.time() - start_time_demand
print(f"✓ Demand-based optimization complete ({elapsed_demand:.1f}s, {len(two_truck_demand_df)} scenarios)")
print(f"\n📊 Best Demand-Based Result:")
print(f"  Trucks: {best_demand['truck_pair']}")
print(f"  Capacity split: {best_demand['cap_split']} entrees")
print(f"  Entrees delivered: {int(best_demand['total_entrees'])}")
print(f"  Net profit: ${best_demand['net_profit']:,.2f}")
print(f"  Total miles: {best_demand['travel_miles']:.1f}")
print(f"  Revenue per mile: ${best_demand['total_revenue'] / best_demand['travel_miles']:.2f}")

   📍 No distance cache provided; using raw profit for selection
   📍 No distance cache provided; using raw profit for selection
   📍 No distance cache provided; using raw profit for selection
   📍 No distance cache provided; using raw profit for selection
   📍 No distance cache provided; using raw profit for selection
   📍 No distance cache provided; using raw profit for selection
   📍 No distance cache provided; using raw profit for selection
   📍 No distance cache provided; using raw profit for selection
   📍 No distance cache provided; using raw profit for selection
   📍 No distance cache provided; using raw profit for selection
   📍 No distance cache provided; using raw profit for selection
   📍 No distance cache provided; using raw profit for selection
   📍 No distance cache provided; using raw profit for selection
   📍 No distance cache provided; using raw profit for selection
   📍 No distance cache provided; using raw profit for selection
   📍 No distance cache provided; using r

In [592]:
# ── Three-Way Comparison: Latitude vs K-Means vs Demand-Based ─────────────
# Format results from all three optimization approaches for comparison

# Extract from latitude split results
best_latitude_formatted = {
    'truck_pair': f"{two_truck_df.iloc[0]['truck1']} + {two_truck_df.iloc[0]['truck2']}",
    'total_entrees': int(two_truck_df.iloc[0]['truck1_entrees'] + two_truck_df.iloc[0]['truck2_entrees']),
    'travel_miles': two_truck_df.iloc[0]['truck1_miles'] + two_truck_df.iloc[0]['truck2_miles'],
    'net_profit': two_truck_df.iloc[0]['combined_profit']
}

# Extract from k-means cluster results
best_cluster_formatted = {
    'truck_pair': f"{two_truck_cluster_df.iloc[0]['truck1']} + {two_truck_cluster_df.iloc[0]['truck2']}",
    'total_entrees': int(two_truck_cluster_df.iloc[0]['truck1_entrees'] + two_truck_cluster_df.iloc[0]['truck2_entrees']),
    'travel_miles': two_truck_cluster_df.iloc[0]['truck1_miles'] + two_truck_cluster_df.iloc[0]['truck2_miles'],
    'net_profit': two_truck_cluster_df.iloc[0]['combined_profit']
}

# best_demand already has the right structure

comparison_data = {
    'Split Strategy': ['Latitude (North/South)', 'K-Means Geographic', 'Demand-Based'],
    'Trucks': [best_latitude_formatted['truck_pair'], best_cluster_formatted['truck_pair'], best_demand['truck_pair']],
    'Entrees': [best_latitude_formatted['total_entrees'], best_cluster_formatted['total_entrees'], int(best_demand['total_entrees'])],
    'Miles': [best_latitude_formatted['travel_miles'], best_cluster_formatted['travel_miles'], best_demand['travel_miles']],
    'Net Profit': [best_latitude_formatted['net_profit'], best_cluster_formatted['net_profit'], best_demand['net_profit']]
}

three_way_df = pd.DataFrame(comparison_data).sort_values('Net Profit', ascending=False).reset_index(drop=True)
three_way_df.index = three_way_df.index + 1
three_way_df['Rank'] = range(1, len(three_way_df) + 1)

# Format currency columns
three_way_df['Net Profit'] = three_way_df['Net Profit'].apply(lambda x: f"${x:,.2f}")
three_way_df['Miles'] = three_way_df['Miles'].apply(lambda x: f"{x:.1f}")

print("🏆 Strategy Comparison Summary")
print("=" * 100)
print(three_way_df[['Rank', 'Split Strategy', 'Trucks', 'Entrees', 'Miles', 'Net Profit']].to_string())
print("=" * 100)

# Calculate differentials
profit_vals = [best_latitude_formatted['net_profit'], 
               best_cluster_formatted['net_profit'], 
               best_demand['net_profit']]
max_profit = max(profit_vals)
min_profit = min(profit_vals)

print(f"\n📈 Insights:")
print(f"  Strategy rankings by profit: {three_way_df['Split Strategy'].tolist()}")
print(f"  Best strategy profit margin: ${max_profit:,.2f}")
print(f"  Profit advantage vs worst: ${max_profit - min_profit:,.2f} ({((max_profit - min_profit) / min_profit * 100):.1f}%)")

strategies = ['Latitude', 'K-Means', 'Demand']
profits = [best_latitude_formatted['net_profit'], best_cluster_formatted['net_profit'], best_demand['net_profit']]

for i, (strat, profit) in enumerate(zip(strategies, profits)):
    advantage = ((profit - min_profit) / min_profit * 100) if min_profit != 0 else 0
    print(f"  {strat:12} → ${profit:>10,.2f} ({advantage:>6.1f}% vs baseline)")

print(f"\n✓ Recommendation: "
      f"Use {'LATITUDE' if best_latitude_formatted['net_profit'] == max_profit else 'K-MEANS' if best_cluster_formatted['net_profit'] == max_profit else 'DEMAND-BASED'} "
      f"split strategy for maximum profitability.")

🏆 Strategy Comparison Summary
   Rank          Split Strategy              Trucks  Entrees Miles Net Profit
1     1            Demand-Based    Albatross/Condor      427  99.1  $8,580.26
2     2  Latitude (North/South)  Albatross + Condor      417  95.1  $8,560.43
3     3      K-Means Geographic  Albatross + Condor      403  83.9  $8,064.68

📈 Insights:
  Strategy rankings by profit: ['Demand-Based', 'Latitude (North/South)', 'K-Means Geographic']
  Best strategy profit margin: $8,580.26
  Profit advantage vs worst: $515.58 (6.4%)
  Latitude     → $  8,560.43 (   6.1% vs baseline)
  K-Means      → $  8,064.68 (   0.0% vs baseline)
  Demand       → $  8,580.26 (   6.4% vs baseline)

✓ Recommendation: Use DEMAND-BASED split strategy for maximum profitability.


In [603]:
print(f"  Strategy Winner:             DEMAND-BASED ALLOCATION 🏆")
print(f"\n🍽️  TOP 3 PROFIT GENERATORS")
print(f"{'─'*100}")
for i in range(3):
    row = restaurant_profit_df.iloc[i]
    print(f"  {i+1}. {row['RESTAURANT']:45s} ${row['Restaurant_Profit']:>10,.2f} ({int(row['ENTREES_ALLOCATED'])} entrees)")
print("\n" + "="*100)

# Determine best overall option including single truck
best_overall_profit = max(best_single_net_profit, best_demand['net_profit'])
if best_single_net_profit >= best_demand['net_profit']:
    best_overall = best_single_truck_dict
    best_overall['net_profit'] = best_single_net_profit
    best_overall['truck_pair'] = best_single_truck_name
    best_overall['cap_split'] = f"{best_single_entrees}"
    best_overall_allocated = single_alloc
else:
    best_overall = best_demand
    best_overall_allocated = best_demand_allocated

# Create charts with thinner bars and margin to prevent label cutoff
fig_strategy = go.Figure(data=[go.Bar(x=['Latitude', 'K-Means', 'Demand'], y=[best_latitude_formatted['net_profit'], best_cluster_formatted['net_profit'], best_demand['net_profit']], marker=dict(color=['#3498db', '#f39c12', '#2ecc71']), width=0.25, text=[f"${p:,.0f}" for p in [best_latitude_formatted['net_profit'], best_cluster_formatted['net_profit'], best_demand['net_profit']]], textposition='outside')])
fig_strategy.update_layout(title="Strategy Comparison", yaxis_title="Net Profit ($)", height=500, showlegend=False, margin=dict(t=100, b=50, l=50, r=50))

print("✅ Charts created")

import datetime

# Calculate display times
max_two_truck_time = max(best_two_truck1_driving_time, best_two_truck2_driving_time)

html = f"""<!DOCTYPE html><html><head><meta charset="UTF-8"><meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Route Optimization Dashboard</title><script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
<style>*{{margin:0;padding:0;box-sizing:border-box}}body{{font-family:'Segoe UI',sans-serif;background:#f5f5f5;color:#333}}
.container{{max-width:1400px;margin:0 auto;padding:20px}}.header{{background:linear-gradient(135deg,#667eea,#764ba2);color:white;padding:40px;border-radius:10px;margin-bottom:30px;text-align:center}}
.header h1{{font-size:2.8em;margin-bottom:10px}}.header p{{font-size:1.1em}}.recommendation-box{{background:linear-gradient(135deg,#2ecc71,#27ae60);color:white;padding:30px;border-radius:10px;margin-bottom:30px;box-shadow:0 4px 12px rgba(0,0,0,0.15)}}
.recommendation-box h2{{font-size:1.6em;margin-bottom:15px}}.rec-item{{margin:10px 0}}.rec-label{{font-weight:600;opacity:0.9}}.rec-value{{font-size:1.1em;font-weight:700;margin-top:3px}}.metrics-grid{{display:grid;grid-template-columns:repeat(auto-fit,minmax(300px,1fr));gap:20px;margin-bottom:30px}}
.metric-card{{background:white;padding:25px;border-radius:10px;box-shadow:0 2px 8px rgba(0,0,0,0.1);border-top:4px solid #667eea}}.metric-card h3{{color:#667eea;margin-bottom:15px}}
.metric-item{{display:flex;justify-content:space-between;padding:10px 0;border-bottom:1px solid #f0f0f0}}.metric-item:last-child{{border-bottom:none}}.metric-label{{color:#666;font-weight:500}}
.metric-value{{color:#333;font-weight:700}}.charts-container{{display:grid;grid-template-columns:1fr;gap:25px;margin-bottom:30px}}.chart-box{{background:white;padding:25px;border-radius:10px;box-shadow:0 2px 8px rgba(0,0,0,0.1)}}
.chart-title{{color:#667eea;font-size:1.4em;margin-bottom:20px;font-weight:600}}.map-container{{width:100%;height:600px;border-radius:10px;overflow:hidden;box-shadow:0 2px 8px rgba(0,0,0,0.1)}}
.map-row{{display:grid;grid-template-columns:1fr 1fr;gap:25px}}.map-row-single{{display:grid;grid-template-columns:1fr}}.restaurants-table{{width:100%;background:white;border-collapse:collapse;border-radius:10px;box-shadow:0 2px 8px rgba(0,0,0,0.1);margin:30px 0}}
.restaurants-table thead{{background:#667eea;color:white}}.restaurants-table th{{padding:15px;text-align:left;font-weight:600}}.restaurants-table td{{padding:12px 15px;border-bottom:1px solid #f0f0f0}}
.restaurants-table tr:hover{{background:#f9f9f9}}.profit-high{{color:#4caf50;font-weight:700}}.profit-medium{{color:#ff9800;font-weight:700}}.profit-low{{color:#f44336;font-weight:700}}
</style></head><body><div class="container"><div class="header"><h1>📊 Delivery Route Optimization</h1><p>ExoticMeatMarkets.com - Profit Maximization Dashboard</p>
<p style="font-size:0.9em;opacity:0.8;">Generated: {datetime.datetime.now().strftime('%B %d, %Y at %I:%M %p')}</p></div>

<div class="recommendation-box">
<h2>✅ Recommended Configuration</h2>
<div class="rec-item"><div class="rec-label">🏆 Best Option</div><div class="rec-value">{best_overall['truck_pair']} - Demand-Based Allocation</div></div>
<div class="rec-item"><div class="rec-label">💰 Highest Net Profit</div><div class="rec-value">${best_overall['net_profit']:,.2f}</div></div>
<div class="rec-item"><div class="rec-label">📦 Allocation</div><div class="rec-value">{best_overall['cap_split']} entrees across {len(best_overall_allocated)} restaurants</div></div>
<div class="rec-item"><div class="rec-label">🎯 Method</div><div class="rec-value">Demand-weighted restaurant selection optimizes profit while minimizing route inefficiency</div></div>
</div>

<div class="metrics-grid">
<div class="metric-card"><h3>💰 Profitability</h3>
<div class="metric-item"><span class="metric-label">Net Profit</span><span class="metric-value">${best_overall['net_profit']:,.2f}</span></div>
<div class="metric-item"><span class="metric-label">Total Revenue</span><span class="metric-value">${best_overall.get('total_revenue', best_demand['total_revenue']):,.2f}</span></div>
<div class="metric-item"><span class="metric-label">Supply Cost</span><span class="metric-value">${best_overall.get('supply_cost', best_demand['supply_cost']):,.2f}</span></div>
<div class="metric-item"><span class="metric-label">Truck Cost</span><span class="metric-value">${best_overall.get('truck_cost', best_demand['truck_cost']):,.2f}</span></div></div>

<div class="metric-card"><h3>🚚 Operations</h3>
<div class="metric-item"><span class="metric-label">Total Entrees</span><span class="metric-value">{int(best_overall_allocated['ENTREES_ALLOCATED'].sum())}</span></div>
<div class="metric-item"><span class="metric-label">Restaurants Served</span><span class="metric-value">{len(best_overall_allocated)}</span></div>
<div class="metric-item"><span class="metric-label">Total Miles</span><span class="metric-value">{best_overall.get('travel_miles', best_demand['travel_miles']):.1f}</span></div></div>

<div class="metric-card"><h3>⏱️ Driving Times</h3>
<div class="metric-item"><span class="metric-label">Single Truck Route</span><span class="metric-value">{best_single_driving_time:.1f} min</span></div>
<div class="metric-item"><span class="metric-label">Truck 1 (Two-Truck)</span><span class="metric-value">{best_two_truck1_driving_time:.1f} min</span></div>
<div class="metric-item"><span class="metric-label">Truck 2 (Two-Truck)</span><span class="metric-value">{best_two_truck2_driving_time:.1f} min</span></div></div>
</div>

<div class="charts-container">
<div class="chart-box"><div class="chart-title">🏆 Split Strategy Comparison</div><div id="chart3"></div></div>
</div>

<h2 style="color:#333;margin:40px 0 20px;font-size:1.8em;">🔥 Demand Heatmap</h2>
<div style="background:white;padding:20px;border-radius:8px;box-shadow:0 2px 8px rgba(0,0,0,0.1);margin-bottom:30px;">
<p style="color:#666;margin:0 0 15px;font-size:0.95em;"><strong>📍 Demand Distribution Map:</strong> Shows the geographic concentration of restaurant demand across Dallas. Hotter colors (red) indicate higher demand areas, while cooler colors (yellow) indicate lower demand. This visualization helps identify market hotspots and delivery route opportunities.</p>
<div><h3 style="text-align:center;color:#667eea;margin-bottom:15px;">Restaurant Demand Concentration Heatmap</h3><iframe src="Demand_Concentration_Map.html" class="map-container" style="border:none;"></iframe></div>
</div>

<h2 style="color:#333;margin:40px 0 20px;font-size:1.8em;">🗺️ Route Maps</h2>
<div class="map-row">
<div><h3 style="text-align:center;color:#667eea;margin-bottom:15px;">Single Truck Route</h3><iframe src="route_single_truck.html" class="map-container" style="border:none;"></iframe></div>
<div><h3 style="text-align:center;color:#667eea;margin-bottom:15px;">Two Truck Routes</h3><iframe src="route_two_trucks.html" class="map-container" style="border:none;"></iframe></div>
</div>

<h2 style="color:#333;margin:40px 0 20px;font-size:1.8em;">⏱️ Route Details & Driving Times</h2>
<div style="display:grid;grid-template-columns:1fr 1fr;gap:25px;margin-bottom:30px;">
<div style="background:white;padding:25px;border-radius:10px;box-shadow:0 2px 8px rgba(0,0,0,0.1);border-left:4px solid #3498db;">
<h3 style="color:#3498db;margin-bottom:20px;font-size:1.1em;">🚗 Single Truck Route - {best_single_truck}</h3>
<div style="display:flex;justify-content:space-between;margin:15px 0;padding-bottom:15px;border-bottom:1px solid #f0f0f0;">
<span style="color:#666;font-weight:600;">Restaurants Served:</span>
<span style="font-weight:700;color:#333;font-size:1.1em;">{len(best_single_route[best_single_route['ENTREES_ALLOCATED'] > 0])}</span>
</div>
<div style="display:flex;justify-content:space-between;margin:15px 0;padding-bottom:15px;border-bottom:1px solid #f0f0f0;">
<span style="color:#666;font-weight:600;">Entrees Delivered:</span>
<span style="font-weight:700;color:#333;font-size:1.1em;">{int(best_single_route['ENTREES_ALLOCATED'].sum())}</span>
</div>
<div style="display:flex;justify-content:space-between;margin:15px 0;padding-bottom:15px;border-bottom:1px solid #f0f0f0;">
<span style="color:#666;font-weight:600;">Total Distance:</span>
<span style="font-weight:700;color:#333;font-size:1.1em;">{display_single_miles:.1f} miles</span>
</div>
<div style="display:flex;justify-content:space-between;margin:15px 0;padding-bottom:15px;border-bottom:1px solid #f0f0f0;">
<span style="color:#666;font-weight:600;">⏱️ Driving Time:</span>
<span style="font-weight:700;color:#2ecc71;font-size:1.2em;">{best_single_driving_time:.1f} minutes</span>
</div>
<div style="display:flex;justify-content:space-between;margin:15px 0;">
<span style="color:#666;font-weight:600;">Route Type:</span>
<span style="font-weight:700;color:#333;">HQ → Stops → HQ</span>
</div>
</div>

<div style="background:white;padding:25px;border-radius:10px;box-shadow:0 2px 8px rgba(0,0,0,0.1);border-left:4px solid #f39c12;">
<h3 style="color:#f39c12;margin-bottom:20px;font-size:1.1em;">🚗🚗 Two Truck Routes - {best_two_truck1} + {best_two_truck2}</h3>
<div style="display:grid;grid-template-columns:1fr 1fr;gap:15px;">
<div>
<div style="background:#f9f9f9;padding:15px;border-radius:8px;margin-bottom:15px;">
<span style="color:#3498db;font-weight:700;font-size:0.95em;">TRUCK 1: {best_two_truck1}</span>
<div style="margin-top:12px;font-size:0.9em;">
<div style="display:flex;justify-content:space-between;margin:8px 0;"><span>Stops:</span><span style="font-weight:700;">{len(best_two_route1[best_two_route1['ENTREES_ALLOCATED'] > 0])}</span></div>
<div style="display:flex;justify-content:space-between;margin:8px 0;"><span>Distance:</span><span style="font-weight:700;">{display_truck1_miles:.1f} miles</span></div>
<div style="display:flex;justify-content:space-between;margin:8px 0;color:#2ecc71;"><span style="font-weight:600;">⏱️ Time:</span><span style="font-weight:700;font-size:1.05em;">{best_two_truck1_driving_time:.1f} min</span></div>
</div>
</div>
</div>
<div>
<div style="background:#f9f9f9;padding:15px;border-radius:8px;margin-bottom:15px;">
<span style="color:#2ecc71;font-weight:700;font-size:0.95em;">TRUCK 2: {best_two_truck2}</span>
<div style="margin-top:12px;font-size:0.9em;">
<div style="display:flex;justify-content:space-between;margin:8px 0;"><span>Stops:</span><span style="font-weight:700;">{len(best_two_route2[best_two_route2['ENTREES_ALLOCATED'] > 0])}</span></div>
<div style="display:flex;justify-content:space-between;margin:8px 0;"><span>Distance:</span><span style="font-weight:700;">{display_truck2_miles:.1f} miles</span></div>
<div style="display:flex;justify-content:space-between;margin:8px 0;color:#2ecc71;"><span style="font-weight:600;">⏱️ Time:</span><span style="font-weight:700;font-size:1.05em;">{best_two_truck2_driving_time:.1f} min</span></div>
</div>
</div>
</div>
</div>
<div style="background:linear-gradient(135deg,#ecf0f1,#f9f9f9);padding:15px;border-radius:8px;margin-top:15px;border:2px solid #95a5a6;">
<span style="font-weight:700;color:#34495e;">⏱️ Maximum Delivery Time:</span>
<span style="font-weight:700;color:#e74c3c;font-size:1.15em;margin-left:10px;">{display_two_truck_total_time:.1f} minutes</span>
<span style="display:block;font-size:0.85em;color:#7f8c8d;margin-top:8px;font-style:italic;">Both trucks will complete deliveries within this timeframe</span>
</div>
</div>
</div>

<h2 style="color:#333;margin:40px 0 20px;font-size:1.8em;">🍽️ Top 15 Most Profitable Restaurants</h2>
<table class="restaurants-table"><thead><tr><th>Rank</th><th>Restaurant</th><th>Entrees</th><th>Bid</th><th>Revenue</th><th>Cost</th><th>Profit</th></tr></thead><tbody>
"""

for idx, row in top_15_restaurants.iterrows():
    profit = row['Restaurant_Profit']
    profit_class = 'profit-high' if profit > 300 else ('profit-medium' if profit > 150 else 'profit-low')
    html += f"<tr><td>{idx+1}</td><td>{row['RESTAURANT']}</td><td>{int(row['ENTREES_ALLOCATED'])}</td><td>${row['ENTREE_BID_PRICE']:.2f}</td><td>${row['Revenue']:.2f}</td><td>${row['Supply_Cost']:.2f}</td><td class='{profit_class}'>${profit:.2f}</td></tr>"

html += f"""</tbody></table>

<h2 style="color:#333;margin:40px 0 20px;font-size:1.8em;">⚠️ Skipped Restaurants Analysis</h2>
<div style="background:#fff3cd;border-left:4px solid #ffc107;padding:20px;border-radius:8px;margin:20px 0;">
<p style="color:#856404;margin:0;"><strong>📌 Note:</strong> {len(skipped_restaurants)} restaurants were not included in the optimal delivery route due to capacity constraints or lower profitability. Below are the most profitable opportunities that were skipped.</p>
</div>

<div style="display:grid;grid-template-columns:1fr 1fr;gap:20px;margin-bottom:30px;">
<div style="background:white;padding:20px;border-radius:8px;box-shadow:0 2px 8px rgba(0,0,0,0.1);">
<h3 style="color:#e74c3c;margin-bottom:15px;font-size:1.1em;">📊 Skipped Summary</h3>
<div style="text-align:center;">
<div style="font-size:2.2em;font-weight:700;color:#e74c3c;margin-bottom:10px;">{len(skipped_restaurants)}</div>
<div style="color:#666;margin-bottom:20px;">Restaurants Skipped</div>
<div style="background:#f9f9f9;padding:15px;border-radius:8px;margin-bottom:15px;">
<div style="text-align:left;">
<div style="display:flex;justify-content:space-between;margin:8px 0;"><span style="color:#666;">Entrees Missed:</span><span style="font-weight:700;">{int(skipped_analysis_sorted['ENTREES_REQUESTED'].sum())}</span></div>
<div style="display:flex;justify-content:space-between;margin:8px 0;"><span style="color:#666;">Potential Revenue:</span><span style="font-weight:700;">${skipped_analysis_sorted['POTENTIAL_REVENUE'].sum():,.2f}</span></div>
<div style="display:flex;justify-content:space-between;margin:8px 0;"><span style="color:#666;">Missed Profit:</span><span style="font-weight:700;color:#e74c3c;">${skipped_analysis_sorted['POTENTIAL_PROFIT'].sum():,.2f}</span></div>
</div>
</div>
<div style="background:#e8f4f8;padding:10px;border-radius:6px;font-size:0.9em;color:#01579b;">
<strong>Why Skipped:</strong> Supply cap reached, lower margins, or poor geographic efficiency
</div>
</div>
</div>

<div style="background:white;padding:20px;border-radius:8px;box-shadow:0 2px 8px rgba(0,0,0,0.1);">
<h3 style="color:#f39c12;margin-bottom:15px;font-size:1.1em;">💡 Top Opportunity Cost</h3>
<div style="background:linear-gradient(135deg,#fff3e0,#ffe0b2);padding:15px;border-radius:8px;margin-bottom:15px;">
<div style="font-size:1.2em;font-weight:700;color:#e65100;margin-bottom:5px;">
{skipped_analysis_sorted.iloc[0]['RESTAURANT'] if len(skipped_analysis_sorted) > 0 else 'N/A'}
</div>
<div style="color:#bf360c;margin-bottom:10px;">
Missed ${skipped_analysis_sorted.iloc[0]['POTENTIAL_PROFIT']:,.2f} in profit
</div>
<div style="font-size:0.85em;color:#666;line-height:1.5;">
📦 {int(skipped_analysis_sorted.iloc[0]['ENTREES_REQUESTED'])} entrees @ ${skipped_analysis_sorted.iloc[0]['ENTREE_BID_PRICE']:.2f}/entree
</div>
</div>
</div>
</div>

<table class="restaurants-table"><thead><tr><th>Rank</th><th>Restaurant</th><th>Requested</th><th>Bid Price</th><th>Revenue</th><th>Cost</th><th>Missed Profit</th></tr></thead><tbody>
"""

# Add skipped restaurants to the table
for idx, row in skipped_analysis_sorted.head(10).iterrows():
    html += f"<tr><td>{idx+1}</td><td>{row['RESTAURANT']}</td><td>{int(row['ENTREES_REQUESTED'])}</td><td>${row['ENTREE_BID_PRICE']:.2f}</td><td>${row['POTENTIAL_REVENUE']:,.2f}</td><td>${row['TOTAL_COST']:,.2f}</td><td style='color:#e74c3c;font-weight:700;'>${row['MISSED_PROFIT']:,.2f}</td></tr>"

html += f"""</tbody></table>

<div style="text-align:center;color:#666;margin-top:30px;padding-top:20px;border-top:1px solid #ddd;">
<p>All charts and metrics generated by ExoticMeatMarkets Delivery Optimization</p></div></div>
<script>
var fig3 = {fig_strategy.to_json()}; Plotly.newPlot('chart3', fig3.data, fig3.layout, {{responsive:true}});
</script></body></html>"""

html_file = os.path.join(os.path.expanduser('~'), 'Downloads', 'delivery_dashboard.html')
with open(html_file, 'w', encoding='utf-8') as f:
    f.write(html)

print("✅ DASHBOARD EXPORTED TO HTML")
print(f"📁 File: {html_file}")
print(f"📊 Dashboard exported successfully!")

  Strategy Winner:             DEMAND-BASED ALLOCATION 🏆

🍽️  TOP 3 PROFIT GENERATORS
────────────────────────────────────────────────────────────────────────────────────────────────────
  1. Ruth's Chris Cuy House                        $    700.00 (35 entrees)
  2. Al Biernat's Fine Cuy                         $    602.00 (14 entrees)
  3. The Capital Cuy                               $    600.00 (15 entrees)

✅ Charts created
✅ DASHBOARD EXPORTED TO HTML
📁 File: C:\Users\samdu\Downloads\delivery_dashboard.html
📊 Dashboard exported successfully!


In [600]:
# ── Analyze Skipped Restaurants ──────────────────────────────────────────────
# Identify which restaurants were not allocated in the best scenario and analyze why

# Get allocated restaurants
allocated_restaurants = set(best_overall_allocated['RESTAURANT'].unique())
all_restaurants_set = set(df_raw['RESTAURANT'].unique())

# Find skipped restaurants
skipped_restaurants = all_restaurants_set - allocated_restaurants
print(f"\n📊 Skipped Restaurants Analysis")
print(f"{'='*100}")
print(f"  Total restaurants available: {len(all_restaurants_set)}")
print(f"  Restaurants allocated: {len(allocated_restaurants)}")
print(f"  Restaurants skipped: {len(skipped_restaurants)}")

# Create detailed analysis of skipped restaurants
skipped_analysis = df_raw[df_raw['RESTAURANT'].isin(skipped_restaurants)].copy()
skipped_analysis['TOTAL_COST'] = skipped_analysis['ENTREES_REQUESTED'] * SUPPLY_COST_PER_ENTREE
skipped_analysis['POTENTIAL_REVENUE'] = skipped_analysis['ENTREES_REQUESTED'] * skipped_analysis['ENTREE_BID_PRICE']
skipped_analysis['POTENTIAL_PROFIT'] = skipped_analysis['POTENTIAL_REVENUE'] - skipped_analysis['TOTAL_COST']
skipped_analysis['MISSED_PROFIT'] = skipped_analysis['POTENTIAL_PROFIT'].copy()

# Sort by potential profit (highest first) to show what we missed
skipped_analysis_sorted = skipped_analysis.sort_values('POTENTIAL_PROFIT', ascending=False).reset_index(drop=True)

print(f"\n  Total potential profit from skipped: ${skipped_analysis_sorted['POTENTIAL_PROFIT'].sum():,.2f}")
print(f"  Total entrees not delivered: {int(skipped_analysis_sorted['ENTREES_REQUESTED'].sum())} entrees")
print(f"\n  Top 5 Most Profitable Skipped Restaurants:")
for idx, row in skipped_analysis_sorted.head(5).iterrows():
    print(f"    {idx+1}. {row['RESTAURANT']:40s} - ${row['POTENTIAL_PROFIT']:>8,.2f} ({int(row['ENTREES_REQUESTED'])} entrees)")
    print(f"       Bid: ${row['ENTREE_BID_PRICE']:>6.2f}/entree | Revenue: ${row['POTENTIAL_REVENUE']:>8,.2f}")



📊 Skipped Restaurants Analysis
  Total restaurants available: 32
  Restaurants allocated: 25
  Restaurants skipped: 7

  Total potential profit from skipped: $-244.00
  Total entrees not delivered: 143 entrees

  Top 5 Most Profitable Skipped Restaurants:
    1. Cuyjeda's                                - $   90.00 (15 entrees)
       Bid: $ 58.00/entree | Revenue: $  870.00
    2. Cuy Rosso                                - $   66.00 (22 entrees)
       Bid: $ 55.00/entree | Revenue: $1,210.00
    3. Vector Brewing Pizza and Cuy             - $   36.00 (18 entrees)
       Bid: $ 54.00/entree | Revenue: $  972.00
    4. Pappasito's Cuytina                      - $    0.00 (28 entrees)
       Bid: $ 52.00/entree | Revenue: $1,456.00
    5. Sweet Georgia Cuy                        - $  -80.00 (20 entrees)
       Bid: $ 48.00/entree | Revenue: $  960.00


## Phase 11: Financial Analysis & Results Comparison

Generate detailed summary tables, interactive Plotly visualizations, and a comprehensive HTML dashboard comparing all scenarios. Export final results to Downloads folder for driver handoff.

In [594]:
fig_demand.update_layout(
    title="<b>DEMAND-BASED TWO-TRUCK SCENARIOS (Top 15 by profit)</b>",
    height=500,
    margin=dict(l=10, r=10, t=50, b=10)
)

fig_demand.show()

# ═════════════════════════════════════════════════════════════════════════════
# SECTION 5: CROSS-STRATEGY COMPARISON
# ═════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 90)
print("5️⃣  OVERALL BEST SCENARIOS BY STRATEGY")
print("=" * 90)

best_single = single_df.iloc[0]
best_latitude = two_truck_df.iloc[0]
best_kmeans = two_truck_cluster_df.iloc[0]
best_demand_strategy = two_truck_demand_df.iloc[0]

best_single_profit = float(best_single['net_profit'])
best_latitude_profit = float(best_latitude['combined_profit'])
best_kmeans_profit = float(best_kmeans['combined_profit'])
best_demand_profit = float(best_demand_strategy['net_profit'])

comparison_strategies = pd.DataFrame({
    'Strategy': ['Single-Truck (Best)', 'Latitude Split (Best)', 'K-Means (Best)', 'Demand-Based (Best)'],
    'Configuration': [
        best_single['truck'],
        f"{best_latitude['truck1']} + {best_latitude['truck2']}",
        f"{best_kmeans['truck1']} + {best_kmeans['truck2']}",
        best_demand_strategy['truck_pair']
    ],
    'Entrees': [
        int(best_single['entrees']),
        int(best_latitude['truck1_entrees'] + best_latitude['truck2_entrees']),
        int(best_kmeans['truck1_entrees'] + best_kmeans['truck2_entrees']),
        int(best_demand_strategy['total_entrees'])
    ],
    'Distance (mi)': [
        f"{best_single['miles']:.1f}",
        f"{best_latitude['truck1_miles'] + best_latitude['truck2_miles']:.1f}",
        f"{best_kmeans['truck1_miles'] + best_kmeans['truck2_miles']:.1f}",
        f"{best_demand_strategy['travel_miles']:.1f}"
    ],
    'Net Profit': [
        best_single_profit,
        best_latitude_profit,
        best_kmeans_profit,
        best_demand_profit
    ]
})

fig_comparison = go.Figure(data=[go.Table(
    header=dict(
        values=['<b>Strategy</b>', '<b>Configuration</b>', '<b>Entrees</b>', '<b>Distance</b>', '<b>Net Profit</b>'],
        fill_color='#2F4F4F',
        align='center',
        font=dict(color='white', size=12, family='monospace')
    ),
    cells=dict(
        values=[
            comparison_strategies['Strategy'],
            comparison_strategies['Configuration'],
            comparison_strategies['Entrees'],
            comparison_strategies['Distance (mi)'],
            ['${:,.2f}'.format(x) for x in comparison_strategies['Net Profit']]
        ],
        fill_color=['#F0F0F0', '#FFFFFF', '#FFFFFF', '#FFFFFF', '#FFFFE0'],
        align='center',
        font=dict(size=11, family='monospace'),
        height=35
    )
)])

fig_comparison.update_layout(
    title="<b>WINNER FROM EACH OPTIMIZATION STRATEGY</b>",
    height=250,
    margin=dict(l=10, r=10, t=50, b=10)
)

fig_comparison.show()

# Summary statistics
print("\n📊 SUMMARY STATISTICS:")
print(f"  Single-truck scenarios tested: {len(single_df)}")
print(f"  Latitude-split scenarios tested: {len(two_truck_df)}")
print(f"  K-means scenarios tested: {len(two_truck_cluster_df)}")
print(f"  Demand-based scenarios tested: {len(two_truck_demand_df)}")
print(f"  Total scenarios: {len(single_df) + len(two_truck_df) + len(two_truck_cluster_df) + len(two_truck_demand_df)}")

# Find overall winner
profits = [best_single_profit, best_latitude_profit, best_kmeans_profit, best_demand_profit]
winner_idx = profits.index(max(profits))
print(f"\n🏆 OVERALL WINNER: {comparison_strategies['Configuration'].iloc[winner_idx]}")
print(f"   Net Profit: ${max(profits):,.2f}")
print(f"   Strategy: {comparison_strategies['Strategy'].iloc[winner_idx]}")

print("\n" + "=" * 90)


5️⃣  OVERALL BEST SCENARIOS BY STRATEGY



📊 SUMMARY STATISTICS:
  Single-truck scenarios tested: 5
  Latitude-split scenarios tested: 160
  K-means scenarios tested: 400
  Demand-based scenarios tested: 160
  Total scenarios: 725

🏆 OVERALL WINNER: Condor
   Net Profit: $8,856.33
   Strategy: Single-Truck (Best)



In [595]:
print(f"   • Top 15 latitude-split two-truck options")
print(f"   • Top 15 k-means clustering two-truck options")
print(f"   • Top 15 demand-based two-truck options")
print(f"\n💡 Open dashboard: {html_file_path}")
print("=" * 90)

   • Top 15 latitude-split two-truck options
   • Top 15 k-means clustering two-truck options
   • Top 15 demand-based two-truck options

💡 Open dashboard: C:\Users\samdu\Downloads\ExoticMeatMarkets_Dashboard.html


In [596]:
# DEMAND CONCENTRATION HEAT MAP - Geographic visualization of order volume
# Shows where the highest entree request volumes are concentrated

import folium
from folium.plugins import HeatMap, MarkerCluster

# Use all restaurants from df_raw (allocated + skipped), excluding those without coordinates
all_restaurants = df_raw.dropna(subset=['LATITUDE', 'LONGITUDE']).copy()

print(f"📍 Processing {len(all_restaurants)} restaurants with coordinates")
if len(all_restaurants) < len(df_raw):
    print(f"   (Skipped {len(df_raw) - len(all_restaurants)} restaurants without geocoded coordinates)")

# Create heat map centered on Dallas
demand_map = folium.Map(
    location=[HQ_LAT, HQ_LON],
    zoom_start=11,
    tiles='OpenStreetMap'
)

# Create MarkerCluster group for combining close circles when zoomed out
demand_cluster = MarkerCluster(name='Demand Clusters').add_to(demand_map)

# Prepare data for heat layer - using ENTREES_REQUESTED for demand concentration
max_demand = all_restaurants['ENTREES_REQUESTED'].max()
min_demand = all_restaurants['ENTREES_REQUESTED'].min()
demand_range = max_demand - min_demand if max_demand > min_demand else 1

heat_data = []
for _, row in all_restaurants.iterrows():
    # Normalize demand to 0-1 scale
    normalized_demand = (row['ENTREES_REQUESTED'] - min_demand) / demand_range if demand_range > 0 else 0.5
    heat_data.append([row['LATITUDE'], row['LONGITUDE'], normalized_demand])

# Add heat layer with demand intensity
HeatMap(heat_data, radius=30, blur=25, max_zoom=1, min_opacity=0.3).add_to(demand_map)

# Color function for demand: Yellow (low) → Red (high)
def get_demand_color(demand, min_val, max_val):
    """
    Returns a color based on demand volume.
    Yellow (low demand) → Red (high demand)
    """
    if max_val == min_val:
        normalized = 0.5
    else:
        normalized = (demand - min_val) / (max_val - min_val)
    
    # Map normalized [0,1] to yellow-to-red spectrum
    # Yellow is (255, 255, 0), Red is (255, 0, 0)
    r = 255
    g = int(255 * (1 - normalized))  # 255 at low demand, 0 at high demand
    b = 0
    
    return f'rgb({r}, {g}, {b})'

# Build allocation map from best single truck route for comparison
# Get restaurants that were allocated (have entries in best_single_route with qty > 0)
allocated_restaurants_set = set(best_single_route[best_single_route['ENTREES_ALLOCATED'] > 0]['RESTAURANT'].unique())

for _, row in all_restaurants.iterrows():
    demand = row['ENTREES_REQUESTED']
    is_allocated = row['RESTAURANT'] in allocated_restaurants_set
    
    # Circle size proportional to demand
    radius = 5 + (demand - min_demand) / demand_range * 20 if demand_range > 0 else 12
    
    if is_allocated:
        # Allocated = Yellow to Red gradient (demand satisfied)
        color = get_demand_color(demand, min_demand, max_demand)
        popup_prefix = '<b style="color: green;">✓ ALLOCATED</b>'
        allocated_units = best_single_route[best_single_route['RESTAURANT'] == row['RESTAURANT']]['ENTREES_ALLOCATED'].values[0]
        unmet = int(demand - allocated_units)
        unmet_str = f'<br>Unmet demand: {unmet}' if unmet > 0 else ''
    else:
        # Skipped = Red/Orange blend (demand NOT met)
        color = f'rgb(255, {int(100 + (demand - min_demand) / demand_range * 100)}, 50)'  # Red-orange gradient
        popup_prefix = '<b style="color: darkred;">✗ SKIPPED</b>'
        allocated_units = 0
        unmet_str = f'<br>Unmet demand: {int(demand)}'
    
    popup_html = f"""
    {popup_prefix}<br>
    <b>{row['RESTAURANT']}</b><br>
    ─────────────────<br>
    Total Demand: {int(demand)} entrees<br>
    Unit Price: ${row['ENTREE_BID_PRICE']:.2f}<br>
    Allocated: {int(allocated_units)} entrees{unmet_str}
    """
    
    folium.CircleMarker(
        location=[row['LATITUDE'], row['LONGITUDE']],
        radius=radius,
        popup=folium.Popup(popup_html, max_width=300),
        color=color,
        fill=True,
        fillColor=color,
        fillOpacity=0.8,
        weight=2,
        opacity=0.9
    ).add_to(demand_cluster)

# Add HQ marker
folium.Marker(
    location=[HQ_LAT, HQ_LON],
    popup='HQ - 8333 George Coker Circle',
    icon=folium.Icon(color='darkpurple', icon='home'),
    tooltip='Headquarters'
).add_to(demand_map)

# Add legend
total_demand = int(df_raw['ENTREES_REQUESTED'].sum())
allocated_demand = int(all_restaurants[all_restaurants['RESTAURANT'].isin(allocated_restaurants_set)]['ENTREES_REQUESTED'].sum())
unmet_demand = total_demand - allocated_demand

# Identify top demand zones
top_demand = all_restaurants.nlargest(5, 'ENTREES_REQUESTED')

legend_html = f'''
<div style="position: fixed; 
     bottom: 50px; right: 50px; width: 320px; height: auto; 
     background-color: white; border:2px solid grey; z-index:9999; 
     font-size:12px; padding: 15px; border-radius: 5px; box-shadow: 2px 2px 6px rgba(0,0,0,0.3); font-family: Arial, sans-serif;">
     <b style="font-size: 16px;">📊 DEMAND CONCENTRATION</b><br>
     <hr style="margin: 8px 0; border: none; border-top: 1px solid #ccc;">
     
     <b style="color: green;">✓ ALLOCATED</b><br>
     <span style="display:inline-block; width:12px; height:12px; background:rgb(255,255,0); border-radius:50%; margin-right:5px;"></span>🟡 Low Demand<br>
     <span style="display:inline-block; width:12px; height:12px; background:rgb(255,128,0); border-radius:50%; margin-right:5px;"></span>🟠 Medium Demand<br>
     <span style="display:inline-block; width:12px; height:12px; background:rgb(255,0,0); border-radius:50%; margin-right:5px;"></span>🔴 High Demand<br>
     <br>
     
     <b style="color: darkred;">✗ SKIPPED</b><br>
     Red/Orange = Unmet demand<br>
     <br>
     
     <b>DEMAND SUMMARY:</b><br>
     📦 Total demand (all): {total_demand} entrees<br>
     ✓ Allocated: {allocated_demand} entrees<br>
     ✗ Unmet: {unmet_demand} entrees<br>
     <br>
     
     <b>Top 3 Demand Zones:</b><br>
'''

for idx, (_, row) in enumerate(top_demand.head(3).iterrows(), 1):
    pct = (row['ENTREES_REQUESTED'] / total_demand * 100)
    legend_html += f"{idx}. {row['RESTAURANT'][:22]:22} {int(row['ENTREES_REQUESTED']):3}u ({pct:4.1f}%)<br>"

legend_html += '''
     <br>
     <small style="color: #666;">Larger circles = More demand</small>
</div>
'''

demand_map.get_root().html.add_child(folium.Element(legend_html))

# Save the map with relative path
demand_map_file = 'Demand_Concentration_Map.html'
demand_map.save(demand_map_file)

print(f"✅ Demand concentration heat map created!")
print(f"📍 Saved to: {demand_map_file}")
print(f"\n{'='*70}")
print(f"DEMAND CONCENTRATION ANALYSIS".center(70))
print(f"{'='*70}")
print(f"\n📊 Demand Distribution:")
print(f"   Total market demand: {total_demand} entrees")
print(f"   Your capacity (allocated): {allocated_demand} entrees ({allocated_demand/total_demand*100:.1f}%)")
print(f"   Unmet demand: {unmet_demand} entrees ({unmet_demand/total_demand*100:.1f}%)")
print(f"\n🎨 Color Scheme:")
print(f"   🟡 Yellow = Low demand")
print(f"   🟠 Orange = Medium demand")
print(f"   🔴 Red    = High demand")
print(f"\n🔥 Top 5 Demand Generators (by entree requests):\n")

top_5_demand = df_raw.nlargest(5, 'ENTREES_REQUESTED')[['RESTAURANT', 'ENTREES_REQUESTED']]
for idx, (_, row) in enumerate(top_5_demand.iterrows(), 1):
    pct = (row['ENTREES_REQUESTED'] / total_demand * 100)
    is_allocated = row['RESTAURANT'] in allocated_restaurants_set
    status = "✓ ALLOCATED" if is_allocated else "✗ SKIPPED"
    print(f"  {idx}. {row['RESTAURANT']:35} | {int(row['ENTREES_REQUESTED']):3} units ({pct:5.1f}%) [{status}]")

print(f"\n📍 Geographic Clusters (North vs South):")
north = df_raw[df_raw['LATITUDE'] >= df_raw['LATITUDE'].median()]
south = df_raw[df_raw['LATITUDE'] < df_raw['LATITUDE'].median()]
print(f"   North (>= {df_raw['LATITUDE'].median():.2f}°): {int(north['ENTREES_REQUESTED'].sum())} entrees ({north['ENTREES_REQUESTED'].sum()/total_demand*100:.1f}%)")
print(f"   South (< {df_raw['LATITUDE'].median():.2f}°): {int(south['ENTREES_REQUESTED'].sum())} entrees ({south['ENTREES_REQUESTED'].sum()/total_demand*100:.1f}%)")
print(f"{'='*70}")

📍 Processing 32 restaurants with coordinates
✅ Demand concentration heat map created!
📍 Saved to: Demand_Concentration_Map.html

                    DEMAND CONCENTRATION ANALYSIS                     

📊 Demand Distribution:
   Total market demand: 589 entrees
   Your capacity (allocated): 446 entrees (75.7%)
   Unmet demand: 143 entrees (24.3%)

🎨 Color Scheme:
   🟡 Yellow = Low demand
   🟠 Orange = Medium demand
   🔴 Red    = High demand

🔥 Top 5 Demand Generators (by entree requests):

  1. Ruth's Chris Cuy House              |  35 units (  5.9%) [✓ ALLOCATED]
  2. Cuy-Fil-A                           |  30 units (  5.1%) [✓ ALLOCATED]
  3. Pappasito's Cuytina                 |  28 units (  4.8%) [✗ SKIPPED]
  4. Cuy de Brazil                       |  25 units (  4.2%) [✓ ALLOCATED]
  5. Cuysteak House                      |  25 units (  4.2%) [✓ ALLOCATED]

📍 Geographic Clusters (North vs South):
   North (>= 32.82°): 313 entrees (53.1%)
   South (< 32.82°): 276 entrees (46.9%)


## Phase 12: Validation & Assertions

In [597]:
# ── Validation assertions ──────────────────────────────────────────────────────
# TODO: add at least 5 assertions. Starter examples below — keep these and add more.

# 1. No truck carries more than its physical capacity
for result in single_results:
    truck = next(t for t in truck_list if t['truck_name'] == result['truck'])
    assert result['entrees'] <= truck['capacity'], \
        f"{result['truck']} over capacity: {result['entrees']} > {truck['capacity']}"
print('✅ All trucks within capacity')

# 2. No single-truck scenario exceeds the daily supply cap
for result in single_results:
    assert result['entrees'] <= DAILY_SUPPLY_CAP, \
        f"{result['truck']} exceeded supply cap: {result['entrees']} > {DAILY_SUPPLY_CAP}"
print('✅ All scenarios respect daily supply cap')

# 3. validate best single-truck has positive net profit and revenue > profit
best_single = single_df.iloc[0]
assert best_single['net_profit'] > 0, \
    f"{best_single['truck']} has non-positive profit: ${best_single['net_profit']:.2f}"
assert best_single['revenue'] > best_single['net_profit'], \
    f"{best_single['truck']} profit exceeds revenue: ${best_single['net_profit']:.2f} > ${best_single['revenue']:.2f}"
print('✅ Best single-truck scenario has valid profit structure')

# 4. validate two-truck combined cap if you deployed two trucks
best_two = two_truck_df.iloc[0]
total_entrees = best_two['truck1_entrees'] + best_two['truck2_entrees']
assert total_entrees <= DAILY_SUPPLY_CAP, \
    f"Combined entrees for {best_two['truck1']} and {best_two['truck2']} exceed supply cap: {total_entrees} > {DAILY_SUPPLY_CAP}"
print('✅ Two-truck scenario respects daily supply cap')

# 5. validate that best two-truck scenario has reasonable profit
best_two = two_truck_df.iloc[0]
assert best_two['combined_profit'] > 0, \
    f"Two-truck scenario has non-positive profit: ${best_two['combined_profit']:.2f}"
print('✅ Best two-truck scenario has valid profit')

print('✅ All validation assertions passed!')

✅ All trucks within capacity
✅ All scenarios respect daily supply cap
✅ Best single-truck scenario has valid profit structure
✅ Two-truck scenario respects daily supply cap
✅ Best two-truck scenario has valid profit
✅ All validation assertions passed!


In [598]:
two_alloc1 = allocate_supply(df_north, cap_truck1, truck1_dict['capacity'], 
                             hq_node=HQ_NODE, distance_cache=distance_cache, 
                             per_mile_cost=truck1_dict['per_mile_cost'])
best_two_route1 = build_route(two_alloc1, G, HQ_NODE, distance_cache)

two_alloc2 = allocate_supply(df_south, cap_truck2, truck2_dict['capacity'], 
                             hq_node=HQ_NODE, distance_cache=distance_cache, 
                             per_mile_cost=truck2_dict['per_mile_cost'])
best_two_route2 = build_route(two_alloc2, G, HQ_NODE, distance_cache)

   📍 Distance-adjusted selection: 15 restaurants with positive adjusted profit
   📍 Distance-adjusted selection: 13 restaurants with positive adjusted profit


## Phase 13: Interpretation & Final Recommendation

Interpret clustering results and provide final truck recommendation with detailed financial justification. Include sensitivity analysis showing how recommendations change at different supply levels.

## Our Recommendation

*Replace this cell with your group's written recommendation.*

Answer these questions:
1. Which truck (or truck pair) do you recommend for today's supply level, and why?
We recommend using the Condor truck because it has enough capacity to serve all of the restaurants, which is the main reason it has the highest net profit of any single truck (or two truck) option. 
2. What net profit does your recommended configuration produce?
$8856.33
3. Which restaurants did you choose to serve, and which did you skip? Why?
We served 25 restaurants and skipped 7. The seven we skipped had low or negative profit potential, even before accounting for the per mile truck costs. 
4. What trade-offs did you encounter? (e.g., a high-margin restaurant that was
   too far out of the way to justify serving)

5. How does your recommendation change at different supply levels?
   At what supply level (if any) does a different truck or configuration win?
6. How did your group approach the HQ return leg in your routing? Did it materially
   affect your truck selection?


## Submission Checklist

Before submitting, confirm:

- [ ] Notebook runs top-to-bottom without errors
- [ ] `DAILY_SUPPLY_CAP` is set as a single variable (never hardcoded downstream)
- [ ] HQ address is geocoded; all routes start and end at HQ
- [ ] All 5+ assert statements pass
- [ ] At least 3 scenarios evaluated at `DAILY_SUPPLY_CAP = 165` (shown in comparison table)
- [ ] At least one additional supply level tested (e.g., 100 or 120 entrees)
- [ ] Final P&L table for your recommended configuration
- [ ] `Our Recommendation` markdown cell is complete with specific numbers
- [ ] Notebook is committed and pushed to your group's GitHub repo
- [ ] GitHub repo link submitted on Canvas before start of X11D2

**File name:** `X11_Project1_[GroupName].ipynb`  
**Canvas assignment:** Project One — Group Challenge  
**Due:** X11D2 (Thursday), start of class  

> ⚡ On X11D2 your instructor will announce the actual supply available for that day.
> Be ready to update `DAILY_SUPPLY_CAP`, re-run, and present your result within minutes.
